# 🍅 AgriTwin-GH — Disease Progression: Proactive Risk Forecasting

---

## Design Note (Single-Script Justification)
This pipeline is implemented as a single, self-contained Jupyter notebook rather than a package.
Reasons:
1. **Simplicity** — All logic is visible end-to-end; no module imports needed.
2. **Reproducibility** — A single file is easy to share, version-control, and re-run identically.
3. **Colab / Jupyter compatible** — Works locally and on Colab with only `repo_root` change.
4. **Staged development** — Sections A→E can be iterated independently before packaging.

## Pipeline Overview
| Stage | Description |
|-------|-------------|
| **A** | CONFIG dict + seed + GPU detection + path setup |
| **B** | CSV load, validation, gap detection, column aliasing |
| **C** | Chronological time-aware split (no shuffle, no leakage) |
| **D** | Disease threshold logic → rolling exposures → risk index (0–100) → risk labels |
| **E** | Artifact saving (thresholds JSON, split summary JSON, preview CSV, full CSV) |
| **F** | Feature engineering (rolling stats, lags, interactions, night-segmented rolling) |
| **G** | Supervised dataset building — RF tabular + RNN sliding windows |
| **H** | Scaling (no leakage) + feature artifact saving |

> **Sections I onward** (RandomForest training + LSTM/GRU) start in the next prompt.

---

## Quick-Start Instructions

### 1. Data Location
```
data/processed/Greenhouse Indoor Conditions/dindigul_greenhouse_indoor_2024.csv
data/processed/Greenhouse Indoor Conditions/dindigul_greenhouse_indoor_2025.csv
```
Or pass a single combined CSV path in `CONFIG['input_csv_path']`.

**Google Colab:**
```python
from google.colab import drive
drive.mount('/content/drive')
# Then set CONFIG['repo_root'] = '/content/drive/MyDrive/AgriTwin-GH'
```

### 2. Key CONFIG Values
| Key | Default | When to Change |
|-----|---------|----------------|
| `repo_root` | `'.'` | Full path in Colab |
| `input_csv_path` | combined CSV path | Point to your CSV |
| `train_end` | `'2024-10-31'` | Adjust split boundary |
| `val_end` | `'2024-12-31'` | Adjust validation end |
| `fill_gaps_flag` | `True` | False to skip interpolation |
| `mixed_precision` | `False` | True on modern GPUs |

### 3. Output Locations
```
src/agritwin_gh/models/artifacts/dp_<run_id>/   ← all run artifacts (flat, prefixed)
    thresholds_config.json
    data_split_summary.json
    scaler.pkl
    feature_schema.json
    engineered_features_head.csv
    engineered_features_tail.csv
    rf_feature_importance_<disease>.png   ← (Section I, one per disease)
    rnn_training_history.png              ← (Section J)
data/processed/Disease/
    dp_<run_id>_risk_preview.csv          ← last 200 rows sanity check
    dp_<run_id>_full_risk.csv             ← full enriched dataset for Section F+

Model files (flat, prefixed — no subfolders):
    src/agritwin_gh/models/dp_rf_<disease>_<run_id>.joblib   ← RF (Section I)
    src/agritwin_gh/models/dp_rnn_<run_id>.keras             ← LSTM/GRU (Section J)
```
---


## Section A — Configuration & Environment Setup

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# A1 — Import checks + dependency installation (uv only, minimal)
# Run this cell first; all other cells assume these packages are present.
# ─────────────────────────────────────────────────────────────────────────────
import importlib, subprocess, sys

_REQUIRED = {
    "pandas":       "pandas",
    "numpy":        "numpy",
    "sklearn":      "scikit-learn",
    "tensorflow":   "tensorflow",
    "matplotlib":   "matplotlib",
    "tqdm":         "tqdm",
}

_missing = []
for _mod, _pkg in _REQUIRED.items():
    if importlib.util.find_spec(_mod) is None:
        _missing.append(_pkg)

if _missing:
    print(f"[A1] Installing missing packages via uv: {_missing}")
    subprocess.check_call(["uv", "pip", "install"] + _missing)
    print("[A1] Installation complete. Please restart kernel and re-run.")
else:
    print("[A1] All required packages found. No installation needed.")


[A1] All required packages found. No installation needed.


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# A2 — CONFIG — all tunable parameters in ONE place.
# Modify only this cell before running the notebook.
# ─────────────────────────────────────────────────────────────────────────────
import datetime as _dt

_RUN_TIMESTAMP = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    # ── Repo Root ─────────────────────────────────────────────────────────────
    # In Colab, change to: '/content/drive/MyDrive/AgriTwin-GH'
    "repo_root": ".",

    # ── Input Data ────────────────────────────────────────────────────────────
    # Provide a single combined CSV, or the script will auto-combine 2024+2025.
    # Placeholder — update if you have a pre-combined file.
    "input_csv_path": None,  # set to a path string to skip auto-combine
    "csv_2024": r"data/processed/Greenhouse Indoor Conditions/dindigul_greenhouse_indoor_2024.csv",
    "csv_2025": r"data/processed/Greenhouse Indoor Conditions/dindigul_greenhouse_indoor_2025.csv",

    # ── Column Names (actual CSV headers) ─────────────────────────────────────
    "datetime_col":  "datetime",
    "col_temp":      "indoor_temp",
    "col_humidity":  "indoor_humidity",
    "col_airvel":    "indoor_air_velocity",
    "col_co2":       "indoor_CO2",
    "col_solar":     "solarradiation",
    "col_day_night": "day_night_flag",
    "col_vpd":       "vpd",
    "col_dew":       "dew_point",

    # ── Gap Handling ──────────────────────────────────────────────────────────
    # True → reindex to full hourly range then interpolate continuous cols
    "fill_gaps_flag": True,

    # ── Chronological Split (date strings, inclusive) ─────────────────────────
    "train_end": "2024-10-31",   # train: 2024-01-01 → 2024-10-31
    "val_end":   "2024-12-31",   # val:   2024-11-01 → 2024-12-31
                                 # test:  2025-01-01 → 2025-12-31

    # ── Sliding Window / Horizon ──────────────────────────────────────────────
    "window_N":  24,             # lookback window (hours) for LSTM input
    "horizon_H": [6, 12, 24, 48],  # forecast horizons (hours ahead)

    # ── RandomForest Params ───────────────────────────────────────────────────
    "rf_params": {
        "n_estimators":    300,
        "max_depth":       12,
        "min_samples_leaf": 5,
        "random_state":    42,
        "n_jobs":          -1,
    },

    # ── RNN Params ────────────────────────────────────────────────────────────
    # model_type: 'lstm' (default) or 'gru'
    # GRU justified if training >> 2x slower on CPU (fewer params, comparable accuracy)
    "rnn_params": {
        "model_type": "lstm",
        "layers":     2,
        "hidden":     128,
        "dropout":    0.25,
    },

    # ── Training Params ───────────────────────────────────────────────────────
    "training_params": {
        "batch_size": 64,
        "lr":         1e-3,
        "patience":   10,   # EarlyStopping patience (epochs)
        "epochs":     80,
    },

    # ── Mixed Precision ───────────────────────────────────────────────────────
    "mixed_precision": False,   # set True on Ampere+ GPUs for speedup

    # ── Risk Label Bins (inclusive) ───────────────────────────────────────────
    # Low: 0–33 | Medium: 34–66 | High: 67–100
    "risk_label_bins": {
        "low":    [0,  33],
        "medium": [34, 66],
        "high":   [67, 100],
    },

    # ── Output Paths (relative to repo_root) ─────────────────────────────────
    # Model files saved FLAT in models_dir — distinguished by prefix:
    #   dp_rf_<disease>_<run_id>.joblib   ← RandomForest, one per disease (Section I)
    #   dp_rnn_<run_id>.keras             ← LSTM/GRU model (Section J)
    # Artifact files saved flat in artifacts_dir/<run_id>/ — no subfolder.
    #   Shared artifacts: no prefix (scaler.pkl, feature_schema.json, …)
    #   RF artifacts:     rf_* prefix  (e.g. rf_feature_importance_late_blight.png)
    #   RNN artifacts:    rnn_* prefix (e.g. rnn_training_history.png)
    "models_dir":              "src/agritwin_gh/models",
    "artifacts_dir":           "src/agritwin_gh/models/artifacts",
    "processed_data_dir":      "data/processed/Disease",

    # ── Run ID ────────────────────────────────────────────────────────────────
    "run_id": f"dp_{_RUN_TIMESTAMP}",

    # ── Global Seed ───────────────────────────────────────────────────────────
    "seed": 42,

    # ── Diseases ──────────────────────────────────────────────────────────────
    "diseases": [
        "late_blight",
        "leaf_mold",
        "powdery_mildew",
        "early_blight",
        "spider_mites",
    ],
}

print(f"[A2] CONFIG loaded.")
print(f"     Run ID  : {CONFIG['run_id']}")
print(f"     Train   : start → {CONFIG['train_end']}")
print(f"     Val     : {CONFIG['train_end']} → {CONFIG['val_end']}")
print(f"     Test    : {CONFIG['val_end']} → end")
print(f"     Diseases: {CONFIG['diseases']}")


[A2] CONFIG loaded.
     Run ID  : dp_20260305_111754
     Train   : start → 2024-10-31
     Val     : 2024-10-31 → 2024-12-31
     Test    : 2024-12-31 → end
     Diseases: ['late_blight', 'leaf_mold', 'powdery_mildew', 'early_blight', 'spider_mites']


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# A3 — Seeds + GPU detection + Mixed Precision
# ─────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import tensorflow as tf
from pathlib import Path

_SEED = CONFIG["seed"]
random.seed(_SEED)
np.random.seed(_SEED)
tf.random.set_seed(_SEED)
os.environ["PYTHONHASHSEED"] = str(_SEED)
print(f"[A3] Global seeds set to {_SEED} (python, numpy, tensorflow)")

# GPU detection
_gpus = tf.config.list_physical_devices("GPU")
if _gpus:
    print(f"[A3] GPU(s) detected: {[g.name for g in _gpus]}")
    for _g in _gpus:
        try:
            tf.config.experimental.set_memory_growth(_g, True)
        except RuntimeError:
            pass
else:
    print("[A3] No GPU detected — running on CPU (training will be slower).")

# Mixed precision (optional; off by default)
if CONFIG["mixed_precision"] and _gpus:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("[A3] Mixed precision ENABLED (float16).")
else:
    print("[A3] Mixed precision DISABLED (float32).")

# TF version log
print(f"[A3] TensorFlow version: {tf.__version__}")


[A3] Global seeds set to 42 (python, numpy, tensorflow)
[A3] No GPU detected — running on CPU (training will be slower).
[A3] Mixed precision DISABLED (float32).
[A3] TensorFlow version: 2.20.0


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# A4 — Output path constants (pathlib) + directory creation
# All paths are resolved relative to repo_root so the notebook works both
# locally (notebook lives in notebooks/ subdir) and in Colab.
#
# Artifact layout (flat — no subfolders inside the run dir):
#   P_ARTIFACTS_DP/thresholds_config.json         ← shared
#   P_ARTIFACTS_DP/feature_schema.json            ← shared
#   P_ARTIFACTS_DP/scaler.pkl                     ← shared
#   P_ARTIFACTS_DP/rf_feature_importance_*.png    ← rf_ prefix
#   P_ARTIFACTS_DP/rnn_training_history.png       ← rnn_ prefix
#
# Model layout (flat in P_MODELS_DIR):
#   P_MODELS_DIR/dp_rf_<disease>_<run_id>.joblib  ← RF models
#   P_MODELS_DIR/dp_rnn_<run_id>.keras            ← LSTM/GRU model
# ─────────────────────────────────────────────────────────────────────────────
# Repo-root detection — three strategies in priority order:
#
# 1. VS Code sets __vsc_ipynb_file__ to the notebook's absolute .ipynb path.
#    The notebook lives at <repo_root>/notebooks/, so parent.parent = repo root.
# 2. Walk up from kernel CWD looking for common project-marker files.
# 3. Hard fallback: use CONFIG['repo_root'] (default ".").

_nb_file = globals().get("__vsc_ipynb_file__", None)

if _nb_file and Path(_nb_file).exists():
    # Strategy 1 — most reliable in VS Code
    _ROOT = Path(_nb_file).resolve().parent.parent
    _detect_method = f"__vsc_ipynb_file__ → {_nb_file}"
else:
    # Strategy 2 — walk up from CWD looking for project markers
    _HERE = Path().resolve()
    _markers = ["pyproject.toml", "setup.py", "setup.cfg", ".git"]
    _ROOT = next(
        (p for p in [_HERE] + list(_HERE.parents)
         if any((p / m).exists() for m in _markers)),
        Path(CONFIG["repo_root"]).resolve(),   # Strategy 3 fallback
    )
    _detect_method = f"CWD walk-up from {_HERE}"

# Sync back so B1 (and any other function using config["repo_root"]) also
# gets the correct absolute path.
CONFIG["repo_root"] = str(_ROOT)

_RUN_ID = CONFIG["run_id"]

# Path constants
P_MODELS_DIR     = _ROOT / CONFIG["models_dir"]
P_ARTIFACTS_RUN  = _ROOT / CONFIG["artifacts_dir"] / _RUN_ID
P_ARTIFACTS_DP   = P_ARTIFACTS_RUN      # flat — no disease_progression/ subfolder

# Data processed path
P_PROCESSED_DATA = _ROOT / CONFIG["processed_data_dir"]   # data/processed/Disease/

# Create all required directories
for _p in [P_MODELS_DIR, P_ARTIFACTS_DP, P_PROCESSED_DATA]:
    _p.mkdir(parents=True, exist_ok=True)

print("[A4] Repo root resolved:")
print(f"     Method           : {_detect_method}")
print(f"     _ROOT            : {_ROOT}")
print(f"     Models dir       : {P_MODELS_DIR}")
print(f"       Model prefix   : dp_rf_<disease>_<run_id>.joblib | dp_rnn_<run_id>.keras")
print(f"     Run artifacts    : {P_ARTIFACTS_DP}")
print(f"       (flat — rf_* prefix for RF artifacts, rnn_* for RNN artifacts)")
print(f"     Processed data   : {P_PROCESSED_DATA}")


[A4] Repo root resolved:
     Method           : __vsc_ipynb_file__ → e:\AgriTwin-GH\notebooks\disease_progression_risk_forecasting.ipynb
     _ROOT            : E:\AgriTwin-GH
     Models dir       : E:\AgriTwin-GH\src\agritwin_gh\models
       Model prefix   : dp_rf_<disease>_<run_id>.joblib | dp_rnn_<run_id>.keras
     Run artifacts    : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754
       (flat — rf_* prefix for RF artifacts, rnn_* for RNN artifacts)
     Processed data   : E:\AgriTwin-GH\data\processed\Disease


## Section B — Data Loading & Validation

Loads CSV(s), validates schema, detects and optionally fills gaps, aliases columns.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# B1 — load_and_validate()
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Internal column aliases: actual CSV header → canonical short name used below
_COL_ALIAS = {
    CONFIG["col_temp"]:      "temp",
    CONFIG["col_humidity"]:  "humidity",
    CONFIG["col_airvel"]:    "air_velocity",
    CONFIG["col_co2"]:       "co2",
    CONFIG["col_solar"]:     "solar_radiation",
    CONFIG["col_day_night"]: "day_night_flag",
    CONFIG["col_vpd"]:       "vpd",
    CONFIG["col_dew"]:       "dew_point",
}

# Continuous columns that should be interpolated when filling gaps
_CONTINUOUS_COLS = ["temp", "humidity", "air_velocity", "co2",
                    "solar_radiation", "vpd", "dew_point"]


def _load_single_csv(path: str, datetime_col: str) -> pd.DataFrame:
    """Read one CSV, parse and sort by datetime, set as index."""
    df = pd.read_csv(path, parse_dates=[datetime_col])
    df = df.sort_values(datetime_col).reset_index(drop=True)
    return df


def load_and_validate(config: dict):
    """
    Load one or two CSVs, validate schema, detect gaps, optionally fill gaps.

    Returns
    -------
    df : pd.DataFrame
        Cleaned, aliased DataFrame with datetime as the index (DatetimeIndex).
    gap_report : dict
        {n_missing, missing_hours (list of Timestamps)}
    """
    dt_col = config["datetime_col"]

    # ── 1. Load CSV(s) ────────────────────────────────────────────────────────
    if config["input_csv_path"] is not None:
        print(f"[B1] Loading single CSV: {config['input_csv_path']}")
        df = _load_single_csv(config["input_csv_path"], dt_col)
    else:
        p24 = Path(config["repo_root"]) / config["csv_2024"]
        p25 = Path(config["repo_root"]) / config["csv_2025"]
        print(f"[B1] Auto-combining two CSVs:\n     {p24}\n     {p25}")
        df24 = _load_single_csv(str(p24), dt_col)
        df25 = _load_single_csv(str(p25), dt_col)
        df = pd.concat([df24, df25], ignore_index=True)
        df = df.sort_values(dt_col).reset_index(drop=True)
        print(f"[B1] Combined shape: {df.shape}")

    # ── 2. Validate required columns ──────────────────────────────────────────
    required_raw = list(_COL_ALIAS.keys()) + [dt_col]
    missing_cols = [c for c in required_raw if c not in df.columns]
    if missing_cols:
        raise ValueError(f"[B1] Missing columns in CSV: {missing_cols}")
    print(f"[B1] All required columns present ✓")

    # ── 3. Set datetime as index ───────────────────────────────────────────────
    df[dt_col] = pd.to_datetime(df[dt_col])
    df = df.set_index(dt_col).sort_index()

    # ── 4. Validate day_night_flag is {0,1} ───────────────────────────────────
    dn_raw = CONFIG["col_day_night"]
    df[dn_raw] = df[dn_raw].astype(int)
    unexpected = set(df[dn_raw].unique()) - {0, 1}
    if unexpected:
        raise AssertionError(f"[B1] day_night_flag contains unexpected values: {unexpected}")
    print(f"[B1] day_night_flag validated as {{0,1}} ✓")

    # ── 5. Alias columns ──────────────────────────────────────────────────────
    df = df.rename(columns=_COL_ALIAS)
    alias_display = {k: v for k, v in _COL_ALIAS.items()}
    print(f"[B1] Column aliases applied: {alias_display}")

    # ── 6. Detect hourly gaps ─────────────────────────────────────────────────
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq="h")
    actual_idx = df.index
    missing_hours = full_idx.difference(actual_idx)
    gap_report = {
        "n_missing": len(missing_hours),
        "missing_hours": [str(t) for t in missing_hours.tolist()],
    }
    if len(missing_hours) > 0:
        print(f"[B1] Gap detection: {len(missing_hours)} missing hours found.")
        print(f"     First 5 missing: {missing_hours[:5].tolist()}")
    else:
        print(f"[B1] Gap detection: No missing hours ✓")

    # ── 7. Optionally fill gaps ───────────────────────────────────────────────
    if config["fill_gaps_flag"] and len(missing_hours) > 0:
        df = df.reindex(full_idx)
        for col in _CONTINUOUS_COLS:
            if col in df.columns:
                df[col] = df[col].interpolate(method="linear", limit_direction="both")
        # Forward-fill (then back-fill) categorical/flag column
        df["day_night_flag"] = df["day_night_flag"].ffill().bfill().astype(int)
        print(f"[B1] Gaps filled via linear interpolation (continuous) + ffill (flag).")
    elif config["fill_gaps_flag"] and len(missing_hours) == 0:
        print(f"[B1] fill_gaps_flag=True but no gaps to fill.")

    # ── 8. Final report ───────────────────────────────────────────────────────
    print(f"\n[B1] FINAL DataFrame summary:")
    print(f"     Shape       : {df.shape}")
    print(f"     Date range  : {df.index.min()} → {df.index.max()}")
    print(f"     Columns     : {list(df.columns)}")
    print(f"     Dtypes      :\n{df.dtypes}")
    print(f"     Nulls (if any): {df.isnull().sum()[df.isnull().sum() > 0].to_dict()}")

    return df, gap_report


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# B2 — Run data loading
# ─────────────────────────────────────────────────────────────────────────────
import time as _time_mod

_t0_total = _time_mod.time()

print("=" * 70)
print("SECTION B: DATA LOADING & VALIDATION")
print("=" * 70)

df_raw, gap_report = load_and_validate(CONFIG)

print(f"\n[B2] Data loading complete in {_time_mod.time() - _t0_total:.1f}s")
print(f"     Rows: {len(df_raw):,}  |  Columns: {df_raw.shape[1]}")


SECTION B: DATA LOADING & VALIDATION
[B1] Auto-combining two CSVs:
     E:\AgriTwin-GH\data\processed\Greenhouse Indoor Conditions\dindigul_greenhouse_indoor_2024.csv
     E:\AgriTwin-GH\data\processed\Greenhouse Indoor Conditions\dindigul_greenhouse_indoor_2025.csv
[B1] Combined shape: (17544, 10)
[B1] All required columns present ✓
[B1] day_night_flag validated as {0,1} ✓
[B1] Column aliases applied: {'indoor_temp': 'temp', 'indoor_humidity': 'humidity', 'indoor_air_velocity': 'air_velocity', 'indoor_CO2': 'co2', 'solarradiation': 'solar_radiation', 'day_night_flag': 'day_night_flag', 'vpd': 'vpd', 'dew_point': 'dew_point'}
[B1] Gap detection: No missing hours ✓
[B1] fill_gaps_flag=True but no gaps to fill.

[B1] FINAL DataFrame summary:
     Shape       : (17544, 9)
     Date range  : 2024-01-01 00:00:00 → 2025-12-31 23:00:00
     Columns     : ['temp', 'humidity', 'air_velocity', 'co2', 'solar_radiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy']
     Dtypes      

## Section C — Time-Aware Chronological Split

Strict date-range split with zero shuffle and zero leakage. Scaler will be fit **only on train** in future sections.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# C1 — split_chronological()
# Strategy: date-range slices on DatetimeIndex — strictly NO shuffle.
# Leakage prevention: scaler must ONLY be fit on df_splits['train'] later.
# ─────────────────────────────────────────────────────────────────────────────

def split_chronological(df: pd.DataFrame, config: dict):
    """
    Split df into train / val / test by date cutoffs.
    No shuffling. No overlaps.

    Parameters
    ----------
    df : pd.DataFrame with DatetimeIndex (sorted ascending).
    config : CONFIG dict with keys 'train_end', 'val_end'.

    Returns
    -------
    splits : dict  {'train': DataFrame, 'val': DataFrame, 'test': DataFrame}
    split_summary : dict  (date ranges + row counts for JSON serialization)
    """
    train_end = pd.Timestamp(config["train_end"])
    val_end   = pd.Timestamp(config["val_end"])

    # Inclusive end for train: everything ≤ train_end (end of day)
    train_end_inc = train_end + pd.Timedelta(hours=23)
    val_end_inc   = val_end   + pd.Timedelta(hours=23)

    df_train = df[df.index <= train_end_inc]
    df_val   = df[(df.index > train_end_inc) & (df.index <= val_end_inc)]
    df_test  = df[df.index > val_end_inc]

    # ── Sanity assertions ─────────────────────────────────────────────────────
    # 1. No overlaps
    assert len(set(df_train.index) & set(df_val.index)) == 0, "Train/Val overlap!"
    assert len(set(df_val.index)   & set(df_test.index)) == 0, "Val/Test overlap!"
    assert len(set(df_train.index) & set(df_test.index)) == 0, "Train/Test overlap!"

    # 2. Monotonic ordering
    if len(df_train) > 0 and len(df_val) > 0:
        assert df_train.index.max() < df_val.index.min(), "Train must end before Val starts!"
    if len(df_val) > 0 and len(df_test) > 0:
        assert df_val.index.max() < df_test.index.min(), "Val must end before Test starts!"

    # ── Build split_summary for JSON ──────────────────────────────────────────
    def _rng(d):
        if len(d) == 0:
            return {"start": None, "end": None}
        return {"start": str(d.index.min()), "end": str(d.index.max())}

    split_summary = {
        "train": {**_rng(df_train), "n_rows": len(df_train)},
        "val":   {**_rng(df_val),   "n_rows": len(df_val)},
        "test":  {**_rng(df_test),  "n_rows": len(df_test)},
        "total": len(df),
    }

    # ── Logging ───────────────────────────────────────────────────────────────
    print("\n[C1] Chronological Split Summary")
    print(f"     {'Split':<8} {'Start':<22} {'End':<22} {'Rows':>7}")
    print(f"     {'-'*65}")
    for split_name, info in [("train", split_summary["train"]),
                              ("val",   split_summary["val"]),
                              ("test",  split_summary["test"])]:
        print(f"     {split_name:<8} {str(info['start']):<22} {str(info['end']):<22} {info['n_rows']:>7,}")
    print(f"     {'TOTAL':<8} {'':<22} {'':<22} {split_summary['total']:>7,}")
    print(f"\n[C1] Leakage check: No overlaps between splits ✓")
    print(f"[C1] NOTE: Normalisation scaler will be fit ONLY on train set in Section F.")

    return {"train": df_train, "val": df_val, "test": df_test}, split_summary


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# C2 — Run chronological split
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION C: TIME-AWARE CHRONOLOGICAL SPLIT")
print("=" * 70)

df_splits, split_summary = split_chronological(df_raw, CONFIG)


SECTION C: TIME-AWARE CHRONOLOGICAL SPLIT

[C1] Chronological Split Summary
     Split    Start                  End                       Rows
     -----------------------------------------------------------------
     train    2024-01-01 00:00:00    2024-10-31 23:00:00      7,320
     val      2024-11-01 00:00:00    2024-12-31 23:00:00      1,464
     test     2025-01-01 00:00:00    2025-12-31 23:00:00      8,760
     TOTAL                                                   17,544

[C1] Leakage check: No overlaps between splits ✓
[C1] NOTE: Normalisation scaler will be fit ONLY on train set in Section F.


## Section D — Disease Threshold Logic, Exposure Counters & Risk Index (0–100)

### Design
For each of the 5 diseases we implement three tiers of computation:
1. **Binary condition** — does each hour satisfy the primary environmental threshold? (0 or 1)
2. **Rolling exposures** — count of qualifying hours in a sliding window (6h, 12h, 24h); also night-only.
3. **Risk index (0–100)** — formula combining exposure count with bounded additive modifiers.

**Risk index formula (general form)**:
$$\text{base} = 100 \times \min\!\left(1,\; \frac{\text{exposure}_{24h}}{\text{required\_hrs}_{equiv}}\right)$$
$$\text{modifier}_i = \min\!\left(\text{boost\_cap}_i,\; f_i(\text{conditions})\right)$$
$$\text{risk} = \min(100,\; \max(0,\; \text{base} + \sum_i \text{modifier}_i))$$

**Risk labels** (configurable via `CONFIG['risk_label_bins']`):
| Label | Range |
|-------|-------|
| `low` | 0 – 33 |
| `medium` | 34 – 66 |
| `high` | 67 – 100 |

In [48]:
# ─────────────────────────────────────────────────────────────────────────────
# D1 — DISEASE_THRESHOLDS dict
# All numeric threshold parameters in one place.
# Risk index weights (boosts) can be tuned here; structure must remain fixed.
# ─────────────────────────────────────────────────────────────────────────────

DISEASE_THRESHOLDS = {

    # ─── Late Blight (Phytophthora infestans) ─────────────────────────────────
    # Favoured by: high RH (≥90%) sustained 6h, low VPD (≤0.4), night conditions,
    #              poor ventilation (low air velocity).
    "late_blight": {
        # Primary condition: hourly flag = 1 when ALL hold
        "rh_min":        90.0,   # humidity >= 90 %
        "vpd_max":        0.40,  # VPD <= 0.40 kPa
        # How many qualifying hours in 24h = "saturated" for base index
        "required_hours_24h": 6,
        # Modifiers (additive, bounded)
        "night_boost":         15,  # added when night exposure is high
        "night_boost_cap":     15,  # max contribution from this modifier
        "low_airflow_thresh":   0.5,  # air_velocity (m/s) below which airflow is "low"
        "low_airflow_boost":   10,  # added when mean_airvel in window < thresh
        "low_airflow_boost_cap": 10,
    },

    # ─── Leaf Mold (Passalora fulva) ──────────────────────────────────────────
    # Favoured by: high RH (≥90%), low VPD (≤0.5), sustained 5h.
    # Still air amplifies risk but is a MODIFIER, not a hard gate —
    # dataset AV min = 0.76 m/s; the original 0.30 m/s cutoff never fired,
    # producing risk_leaf_mold = 0 for all rows.
    "leaf_mold": {
        "rh_min":             90.0,
        "vpd_max":             0.50,
        "required_hours_24h":  5,
        # Low-airflow modifier — 1.5 m/s calibrated to dataset range (0.76–5.54 m/s)
        "low_airflow_thresh":   1.5,
        "low_airflow_boost":   10,
        "low_airflow_boost_cap": 10,
        # Night boost: sustained high-RH nights accelerate sporulation
        "night_boost":         10,
        "night_boost_cap":     10,
    },

    # ─── Powdery Mildew (Oidium neolycopersici) ───────────────────────────────
    # Unusual: prefers moderate RH (70–90%), moderate VPD (0.5–1.2), low airflow.
    # Does NOT require leaf wetness; VPD range is key.
    "powdery_mildew": {
        "rh_min":             70.0,
        "rh_max":             90.0,
        "vpd_min":             0.50,
        "vpd_max":             1.20,
        "required_hours_24h":  6,
        "low_airflow_thresh":   0.5,
        "low_airflow_boost":   10,
        "low_airflow_boost_cap": 10,
    },

    # ─── Early Blight (Alternaria solani) ────────────────────────────────────
    # Favoured by: mild-warm temp (24–30°C), high RH (≥85%) sustained 4h.
    # Day radiation stress accelerates lesion development.
    "early_blight": {
        "temp_min":            24.0,  # °C
        "temp_max":            30.0,
        "rh_min":              85.0,
        "required_hours_24h":   4,
        # Radiation stress modifier
        "radiation_boost_thresh": 200.0,  # W/m² — high midday radiation
        "radiation_boost":        12,     # added when mean daily radiation > thresh
        "radiation_boost_cap":    12,
    },

    # ─── Spider Mites (Tetranychus urticae) ─────────────────────────────────
    # Favoured by: high temp (≥28°C), very low RH (≤55%), high VPD (≥1.5).
    # High solar radiation / heat stress further worsens.
    "spider_mites": {
        "temp_min":            28.0,
        "rh_max":              55.0,
        "vpd_min":              1.50,
        "required_hours_24h":   1,   # even 1h of severe stress is risky
        "radiation_boost_thresh": 150.0,
        "radiation_boost":        15,
        "radiation_boost_cap":    15,
    },
}

print("[D1] DISEASE_THRESHOLDS defined for:", list(DISEASE_THRESHOLDS.keys()))


[D1] DISEASE_THRESHOLDS defined for: ['late_blight', 'leaf_mold', 'powdery_mildew', 'early_blight', 'spider_mites']


In [49]:
# ─────────────────────────────────────────────────────────────────────────────
# D2 — compute_binary_condition()
# Each disease has a primary condition: an hourly boolean True/False (0/1).
# These are the "elementary" favourable-hour flags used in rolling windows.
# ─────────────────────────────────────────────────────────────────────────────

def compute_binary_condition(df: pd.DataFrame, disease: str) -> pd.Series:
    """
    Returns a 0/1 Series indicating whether hour i satisfies the
    primary environmental threshold for `disease`.

    Column assumptions (after aliasing in Section B):
        temp, humidity, air_velocity, solar_radiation, vpd, day_night_flag
    """
    th = DISEASE_THRESHOLDS[disease]

    # Shortcuts for brevity
    T  = df["temp"]
    RH = df["humidity"]
    AV = df["air_velocity"]
    SR = df["solar_radiation"]
    VP = df["vpd"]
    DN = df["day_night_flag"]  # 0=night, 1=day (in this dataset convention)

    if disease == "late_blight":
        # Condition: RH ≥ rh_min AND VPD ≤ vpd_max
        cond = (RH >= th["rh_min"]) & (VP <= th["vpd_max"])

    elif disease == "leaf_mold":
        # Primary condition: RH ≥ rh_min AND VPD ≤ vpd_max
        # Still air (AV) is now a modifier boost in compute_risk_index() — not a hard gate.
        # Rationale: dataset AV min = 0.76 m/s; the original 0.30 m/s cutoff never fired.
        cond = (RH >= th["rh_min"]) & (VP <= th["vpd_max"])

    elif disease == "powdery_mildew":
        # Condition: rh_min ≤ RH ≤ rh_max AND vpd_min ≤ VPD ≤ vpd_max
        cond = (
            (RH >= th["rh_min"]) & (RH <= th["rh_max"])
            & (VP >= th["vpd_min"]) & (VP <= th["vpd_max"])
        )

    elif disease == "early_blight":
        # Condition: temp_min ≤ T ≤ temp_max AND RH ≥ rh_min
        cond = (T >= th["temp_min"]) & (T <= th["temp_max"]) & (RH >= th["rh_min"])

    elif disease == "spider_mites":
        # Condition: T ≥ temp_min AND RH ≤ rh_max AND VPD ≥ vpd_min
        cond = (T >= th["temp_min"]) & (RH <= th["rh_max"]) & (VP >= th["vpd_min"])

    else:
        raise ValueError(f"Unknown disease: {disease}")

    return cond.astype(int)


# Quick smoke test (will run after df_raw is loaded)

print("[D2] compute_binary_condition() defined.")

[D2] compute_binary_condition() defined.


In [50]:
# ─────────────────────────────────────────────────────────────────────────────
# D3 — compute_rolling_exposures()
# For each disease, compute:
#   exposure_count_6h  = qualifying hours in last 6h  (rolling sum)
#   exposure_count_12h = qualifying hours in last 12h
#   exposure_count_24h = qualifying hours in last 24h
#   night_exposure_24h = qualifying night hours in last 24h
#         (night = day_night_flag == 0, i.e. DN == 0)
#
# Rolling windows are min_periods=1 so early rows get partial counts.
# ─────────────────────────────────────────────────────────────────────────────

def compute_rolling_exposures(df: pd.DataFrame, disease: str,
                               windows: list = [6, 12, 24]) -> pd.DataFrame:
    """
    Append rolling exposure columns to df (copy) for `disease`.

    Returns
    -------
    df_out : pd.DataFrame with extra columns:
        cond_<disease>              — binary condition flag (0/1)
        exposure_count_6h_<disease>
        exposure_count_12h_<disease>
        exposure_count_24h_<disease>
        night_exposure_24h_<disease>
    """
    df_out = df.copy()

    # Binary primary condition
    cond = compute_binary_condition(df, disease)
    cond_col = f"cond_{disease}"
    df_out[cond_col] = cond

    # Rolling exposure windows
    for w in windows:
        col = f"exposure_count_{w}h_{disease}"
        df_out[col] = (
            cond.rolling(window=w, min_periods=1).sum().astype(int)
        )

    # Night-only exposure over 24h
    # day_night_flag == 0 → night in this dataset
    night_mask = (df["day_night_flag"] == 0).astype(int)
    night_cond = cond * night_mask
    df_out[f"night_exposure_24h_{disease}"] = (
        night_cond.rolling(window=24, min_periods=1).sum().astype(int)
    )

    return df_out


print("[D3] compute_rolling_exposures() defined.")


[D3] compute_rolling_exposures() defined.


In [51]:
# ─────────────────────────────────────────────────────────────────────────────
# D4 — compute_risk_index()
# Documented formula (implement EXACTLY this structure; weights are tunable):
#
#   BASE:
#     base = 100 * min(1, exposure_24h / required_hours_24h_equiv)
#     → saturates at 100 when exposure_24h >= required_hours_24h_equiv
#
#   MODIFIERS (per disease, additive, bounded):
#     night_boost_contrib   = min(night_boost_cap,
#                               night_boost * (night_exposure_24h / 24))
#     airflow_boost_contrib = min(low_airflow_boost_cap,
#                               low_airflow_boost * (1 - clamp(mean_airvel /
#                                         low_airflow_thresh, 0, 1)))
#     radiation_boost_contrib = min(radiation_boost_cap,
#                               radiation_boost * clamp(
#                                 (mean_solar - radiation_boost_thresh) /
#                                  radiation_boost_thresh, 0, 1))
#
#   FINAL:
#     risk = clamp(base + sum(modifiers), 0, 100)
# ─────────────────────────────────────────────────────────────────────────────

def _clamp(x, lo=0.0, hi=1.0):
    """Scalar clamp."""
    return max(lo, min(hi, x))


def compute_risk_index(df: pd.DataFrame, disease: str) -> pd.Series:
    """
    Compute the continuous risk index (0–100 float) for each hour.

    Assumes df already contains the rolling exposure columns produced by
    compute_rolling_exposures().

    Equations in comments reference the formula block above (D4).
    """
    th = DISEASE_THRESHOLDS[disease]
    req_24h = th["required_hours_24h"]

    exp24  = df[f"exposure_count_24h_{disease}"].values.astype(float)
    night24 = df[f"night_exposure_24h_{disease}"].values.astype(float)

    n = len(df)
    risk = np.zeros(n, dtype=float)

    # Vectorised rolling mean for air_velocity and solar_radiation (24h window)
    av_mean24  = df["air_velocity"].rolling(24, min_periods=1).mean().values
    sr_mean24  = df["solar_radiation"].rolling(24, min_periods=1).mean().values

    for i in range(n):
        # ── BASE ──────────────────────────────────────────────────────────────
        # base = 100 * min(1, exposure_24h / required_hours_24h)
        base = 100.0 * min(1.0, exp24[i] / req_24h)

        modifiers = 0.0

        # ── NIGHT BOOST (late_blight only by spec; structure available to all)
        if "night_boost" in th:
            # night_boost_contrib = min(cap, boost * fraction_of_night_exposure)
            # fraction_of_night_exposure = night_exposure_24h / 24
            night_frac = night24[i] / 24.0
            nb = min(th["night_boost_cap"], th["night_boost"] * night_frac)
            modifiers += nb

        # ── LOW AIRFLOW BOOST (late_blight, powdery_mildew) ──────────────────
        if "low_airflow_thresh" in th:
            # airflow_boost_contrib = min(cap,
            #   boost * (1 - clamp(mean_airvel / threshold, 0, 1)))
            # → maximum boost when mean_airvel → 0; zero when ≥ threshold
            av_ratio = _clamp(av_mean24[i] / th["low_airflow_thresh"])
            ab = min(th["low_airflow_boost_cap"], th["low_airflow_boost"] * (1.0 - av_ratio))
            modifiers += ab

        # ── RADIATION BOOST (early_blight, spider_mites) ─────────────────────
        if "radiation_boost_thresh" in th:
            # radiation_boost_contrib = min(cap,
            #   boost * clamp((mean_solar - thresh) / thresh, 0, 1))
            # → linearly ramps from 0 to boost as solar_radiation goes
            #   from threshold to 2×threshold; capped at max boost.
            sr_excess = _clamp(
                (sr_mean24[i] - th["radiation_boost_thresh"]) / th["radiation_boost_thresh"]
            )
            rb = min(th["radiation_boost_cap"], th["radiation_boost"] * sr_excess)
            modifiers += rb

        # ── FINAL: clamp to [0, 100] ──────────────────────────────────────────
        risk[i] = min(100.0, max(0.0, base + modifiers))

    return pd.Series(risk, index=df.index, name=f"risk_{disease}")


print("[D4] compute_risk_index() defined.")


[D4] compute_risk_index() defined.


In [52]:
# ─────────────────────────────────────────────────────────────────────────────
# D5 — assign_risk_label() + run_disease_risk_pipeline() (orchestrator)
# ─────────────────────────────────────────────────────────────────────────────

def assign_risk_label(risk_series: pd.Series, bins: dict) -> pd.Series:
    """
    Map continuous risk index (0–100) to categorical label.

    bins example (from CONFIG['risk_label_bins']):
        {'low': [0, 33], 'medium': [34, 66], 'high': [67, 100]}

    Returns Series of strings: 'low' | 'medium' | 'high'
    """
    lo_lo, lo_hi  = bins["low"]
    me_lo, me_hi  = bins["medium"]
    hi_lo, hi_hi  = bins["high"]

    def _label(v):
        if lo_lo <= v <= lo_hi:  return "low"
        if me_lo <= v <= me_hi:  return "medium"
        if hi_lo <= v <= hi_hi:  return "high"
        return "high"  # fallback for exact 100

    return risk_series.map(_label)


def run_disease_risk_pipeline(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Orchestrator: iterate all 5 diseases, compute exposures + risk index
    + risk labels, and append columns to df.

    Output columns added per disease:
        cond_<disease>               — binary threshold flag
        exposure_count_6h_<disease>
        exposure_count_12h_<disease>
        exposure_count_24h_<disease>
        night_exposure_24h_<disease>
        risk_<disease>               — float 0–100
        risk_label_<disease>         — 'low' | 'medium' | 'high'

    Returns enriched DataFrame.
    """
    bins = config["risk_label_bins"]
    df_enriched = df.copy()

    print("\n[D5] Running disease risk pipeline...")
    print(f"     {'Disease':<20} {'Min':>6} {'Max':>6} {'Mean':>6} {'Low%':>6} {'Med%':>6} {'Hi%':>6}")
    print(f"     {'-'*60}")

    for disease in config["diseases"]:
        # 1. Rolling exposures (adds cond_ and exposure_count_ columns)
        df_enriched = compute_rolling_exposures(df_enriched, disease)

        # 2. Risk index
        risk = compute_risk_index(df_enriched, disease)
        df_enriched[f"risk_{disease}"] = risk

        # 3. Risk labels
        labels = assign_risk_label(risk, bins)
        df_enriched[f"risk_label_{disease}"] = labels

        # 4. Summary stats for this disease
        _mn  = risk.min()
        _mx  = risk.max()
        _avg = risk.mean()
        _vc  = labels.value_counts(normalize=True) * 100
        _lo  = _vc.get("low",    0.0)
        _me  = _vc.get("medium", 0.0)
        _hi  = _vc.get("high",   0.0)
        print(f"     {disease:<20} {_mn:>6.1f} {_mx:>6.1f} {_avg:>6.1f} {_lo:>5.1f}% {_me:>5.1f}% {_hi:>5.1f}%")

    print(f"\n[D5] Enriched DataFrame shape: {df_enriched.shape}")
    print(f"[D5] Risk columns added: {[c for c in df_enriched.columns if c.startswith('risk_')]}")
    return df_enriched


print("[D5] assign_risk_label() and run_disease_risk_pipeline() defined.")


[D5] assign_risk_label() and run_disease_risk_pipeline() defined.


In [53]:
# ─────────────────────────────────────────────────────────────────────────────
# D6 — Run the risk pipeline on the full dataset
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION D: DISEASE THRESHOLD LOGIC & RISK INDEX")
print("=" * 70)

_t_d = _time_mod.time()
df_enriched = run_disease_risk_pipeline(df_raw, CONFIG)
print(f"\n[D6] Risk pipeline complete in {_time_mod.time() - _t_d:.1f}s")

# Quick sanity peek: last 5 rows showing risk columns
_risk_cols = [c for c in df_enriched.columns if c.startswith("risk_")]
print("\n[D6] Last 5 rows (risk columns only):")
print(df_enriched[_risk_cols].tail(5).to_string())


SECTION D: DISEASE THRESHOLD LOGIC & RISK INDEX

[D5] Running disease risk pipeline...
     Disease                 Min    Max   Mean   Low%   Med%    Hi%
     ------------------------------------------------------------
     late_blight             0.0  100.0   12.0  85.7%   3.5%  10.8%
     leaf_mold               0.0  100.0   13.0  85.7%   3.6%  10.7%
     powdery_mildew          0.0  100.0   72.5  24.1%   2.4%  73.5%
     early_blight            0.0  100.0   26.8  72.4%   2.4%  25.2%
     spider_mites            0.0  100.0   29.0  71.0%   0.0%  29.0%

[D5] Enriched DataFrame shape: (17544, 44)
[D5] Risk columns added: ['risk_late_blight', 'risk_label_late_blight', 'risk_leaf_mold', 'risk_label_leaf_mold', 'risk_powdery_mildew', 'risk_label_powdery_mildew', 'risk_early_blight', 'risk_label_early_blight', 'risk_spider_mites', 'risk_label_spider_mites']

[D6] Risk pipeline complete in 0.5s

[D6] Last 5 rows (risk columns only):
                     risk_late_blight risk_label_late_bli

## Section E — Artifact Saving

Saves four artifacts per run into the structured repo paths defined in Section A:

| Artifact | Path |
|----------|------|
| `thresholds_config.json` | `src/agritwin_gh/models/artifacts/dp_<run_id>/` |
| `data_split_summary.json` | `src/agritwin_gh/models/artifacts/dp_<run_id>/` |
| `dp_<run_id>_risk_preview.csv` | `data/processed/Disease/` (last 200 rows) |
| `dp_<run_id>_full_risk.csv` | `data/processed/Disease/` (full enriched dataset, for Section F) |


In [54]:
# ─────────────────────────────────────────────────────────────────────────────
# E1 — save_artifacts()
# All four artifacts are saved with absolute paths (derived from pathlib roots
# set in Section A). No ambiguity in file location.
# ─────────────────────────────────────────────────────────────────────────────
import json

def save_artifacts(df_enriched: pd.DataFrame,
                   split_summary: dict,
                   config: dict,
                   gap_report: dict):
    """
    Save all Sections A–D artifacts to their designated repo paths.

    Artifacts produced:
    1. thresholds_config.json   → P_ARTIFACTS_DP / thresholds_config.json
    2. data_split_summary.json  → P_ARTIFACTS_DP / data_split_summary.json
    3. preview CSV (last 200 rows) → P_PROCESSED_DATA / dp_<run_id>_risk_preview.csv
    4. full enriched CSV           → P_PROCESSED_DATA / dp_<run_id>_full_risk.csv
    """
    run_id = config["run_id"]
    bins   = config["risk_label_bins"]
    diseases = config["diseases"]

    # ── 1. thresholds_config.json ────────────────────────────────────────────
    thresholds_payload = {
        "run_id":              run_id,
        "diseases":            diseases,
        "thresholds":          DISEASE_THRESHOLDS,
        "risk_label_bins":     bins,
        "exposure_windows_h":  [6, 12, 24],
    }
    p_thresh = P_ARTIFACTS_DP / "thresholds_config.json"
    with open(p_thresh, "w") as f:
        json.dump(thresholds_payload, f, indent=2)
    print(f"[E1] Saved: {p_thresh}")

    # ── 2. data_split_summary.json ───────────────────────────────────────────
    split_payload = {
        "run_id":       run_id,
        "split_config": {
            "train_end": config["train_end"],
            "val_end":   config["val_end"],
        },
        "splits":       split_summary,
        "gap_report":   gap_report,
    }
    p_split = P_ARTIFACTS_DP / "data_split_summary.json"
    with open(p_split, "w") as f:
        json.dump(split_payload, f, indent=2)
    print(f"[E1] Saved: {p_split}")

    # ── 3. Preview CSV (last 200 rows) ───────────────────────────────────────
    sensor_cols  = ["temp", "humidity", "air_velocity", "solar_radiation",
                    "vpd", "co2", "dew_point", "day_night_flag"]
    risk_cols    = [c for c in df_enriched.columns if c.startswith("risk_")]
    preview_cols = [c for c in sensor_cols if c in df_enriched.columns] + risk_cols

    df_preview = df_enriched[preview_cols].tail(200)
    p_preview  = P_PROCESSED_DATA / f"{run_id}_risk_preview.csv"
    df_preview.to_csv(p_preview)
    print(f"[E1] Saved preview ({len(df_preview)} rows): {p_preview}")

    # ── 4. Full enriched CSV for next stage (feature engineering + modelling)
    p_full = P_PROCESSED_DATA / f"{run_id}_full_risk.csv"
    df_enriched.to_csv(p_full)
    print(f"[E1] Saved full enriched CSV ({len(df_enriched):,} rows): {p_full}")

    return {
        "thresholds_config":     str(p_thresh),
        "data_split_summary":    str(p_split),
        "risk_preview_csv":      str(p_preview),
        "full_risk_csv":         str(p_full),
    }


print("[E1] save_artifacts() defined.")


[E1] save_artifacts() defined.


In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# E2 — Run artifact saving (execute save_artifacts)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION E: ARTIFACT SAVING")
print("=" * 70)

_t_e = _time_mod.time()
saved_paths = save_artifacts(df_enriched, split_summary, CONFIG, gap_report)

print(f"\n[E2] All artifacts saved in {_time_mod.time() - _t_e:.1f}s")
print(f"\n[E2] Artifact index:")
for k, v in saved_paths.items():
    print(f"     {k:<25} → {v}")

print(f"\n{'=' * 70}")
print(f"Pipeline Sections A–E complete. Total elapsed: {_time_mod.time() - _t0_total:.1f}s")
print(f"Run ID : {CONFIG['run_id']}")
print(f"{'=' * 70}")
print(f"\nNext: Sections F–G → feature engineering + RandomForest baseline")
print(f"       Sections H–I → LSTM/GRU multi-step risk forecasting")


SECTION E: ARTIFACT SAVING
[E1] Saved: E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\thresholds_config.json
[E1] Saved: E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\data_split_summary.json
[E1] Saved preview (200 rows): E:\AgriTwin-GH\data\processed\Disease\dp_20260305_111754_risk_preview.csv
[E1] Saved full enriched CSV (17,544 rows): E:\AgriTwin-GH\data\processed\Disease\dp_20260305_111754_full_risk.csv

[E2] All artifacts saved in 0.7s

[E2] Artifact index:
     thresholds_config         → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\thresholds_config.json
     data_split_summary        → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\data_split_summary.json
     risk_preview_csv          → E:\AgriTwin-GH\data\processed\Disease\dp_20260305_111754_risk_preview.csv
     full_risk_csv             → E:\AgriTwin-GH\data\processed\Disease\dp_20260305_111754_full_risk.csv

Pipeline Sections A–E complete. T

In [56]:
# ─────────────────────────────────────────────────────────────────────────────
# E3 — Optional: Risk Index Visualisation (sanity check)
# Run this cell to get a quick plot of risk indices over time.
# Requires matplotlib (already checked in A1).
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

_PLOT_FROM = "2024-06-01"   # adjust window for quicker inspection
_PLOT_TO   = "2024-08-31"

_risk_cols_float = [f"risk_{d}" for d in CONFIG["diseases"]]
_df_plot = df_enriched.loc[_PLOT_FROM:_PLOT_TO, _risk_cols_float]

fig, axes = plt.subplots(len(CONFIG["diseases"]), 1,
                         figsize=(16, 3 * len(CONFIG["diseases"])),
                         sharex=True)

_COLOURS = ["#e63946", "#2a9d8f", "#f4a261", "#457b9d", "#e9c46a"]

for ax, disease, col, colour in zip(axes, CONFIG["diseases"], _risk_cols_float, _COLOURS):
    ax.fill_between(_df_plot.index, _df_plot[col],
                    alpha=0.35, color=colour)
    ax.plot(_df_plot.index, _df_plot[col],
            linewidth=0.8, color=colour, label=disease.replace("_", " ").title())
    # Label thresholds
    ax.axhline(CONFIG["risk_label_bins"]["high"][0],
               color="red", linestyle="--", linewidth=0.8, alpha=0.6, label="High threshold (67)")
    ax.axhline(CONFIG["risk_label_bins"]["medium"][0],
               color="orange", linestyle=":", linewidth=0.8, alpha=0.6, label="Med threshold (34)")
    ax.set_ylabel("Risk (0–100)", fontsize=9)
    ax.set_ylim(0, 105)
    ax.legend(loc="upper right", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Disease Risk Indices — {_PLOT_FROM} to {_PLOT_TO}\n(Run: {CONFIG['run_id']})",
             fontsize=11, y=1.01)
fig.tight_layout()

# Save to DP artifacts folder
_p_fig = P_ARTIFACTS_DP / "risk_index_overview.png"
fig.savefig(_p_fig, dpi=100, bbox_inches="tight")
print(f"[E3] Plot saved: {_p_fig}")
plt.show()


[E3] Plot saved: E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\risk_index_overview.png


## Section F — Feature Engineering

Builds a rich feature matrix **X** and a 5-column target matrix **Y** from `df_enriched`.  
Features are assembled in five layers, then combined in a stable column order.

| Layer | Description | Columns |
|-------|-------------|---------|
| **Base sensors** | Raw aliased sensor cols + `day_night_flag` | 8 |
| **Cyclical time** | `hour_sin/cos`, `month_sin/cos` | 4 |
| **Exposure counters** | From Section D (`cond_*`, `exposure_count_*`, `night_exposure_*`) | 5 diseases × 5 |
| **Rolling stats** | mean/max/min/std for each base sensor, windows 6h/12h/24h | 7 vars × 3 win × 4 = 84 |
| **Lag features** | t−1/2/3/6/12/24 for 6 key variables | 6 vars × 6 lags = 36 |
| **Interaction terms** | `temp_x_rh`, `vpd_x_rh`, `dewpoint_spread` | 3 |
| **Night-segmented rolling** | Night-only rolling mean for each base var, windows 6h/12h/24h | 7 vars × 3 win = 21 |

**Target Y** — 5 continuous columns: `risk_<disease>` (float 0–100, produced in Section D).

> NaN created by early lag rows (first 24 rows) is filled with backward-fill so no
> rows are dropped — the model-training stage will handle warm-up period exclusion based on `window_N`.


In [57]:
# ─────────────────────────────────────────────────────────────────────────────
# F1 — Feature Engineering CONFIG + cyclical time feature helper
# All feature-engineering knobs live here; downstream cells read from FE_CONFIG.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

FE_CONFIG = {
    # Continuous sensor variables to transform
    "base_vars": [
        "temp", "humidity", "air_velocity", "co2",
        "solar_radiation", "vpd", "dew_point",
    ],
    # Subset of base_vars used for lag creation (high signal / domain knowledge)
    "lag_vars": [
        "temp", "humidity", "vpd", "air_velocity", "solar_radiation", "dew_point",
    ],
    # Rolling window sizes (hours)
    "roll_windows": [6, 12, 24],
    # Lag offsets (hours behind current time-step)
    "lags": [1, 2, 3, 6, 12, 24],
    # Inherit from global CONFIG
    "diseases":  CONFIG["diseases"],
    "window_N":  CONFIG["window_N"],   # LSTM lookback
    "horizon_H": CONFIG["horizon_H"],  # Forecast horizons
}


def add_cyclical_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Encode hour-of-day and month-of-year as sin/cos pairs.

    Why: circular encoding avoids artificial distance between hour 23 and hour 0,
    and between December and January — critical for climate-driven disease cycles.

    Columns added: hour_sin, hour_cos, month_sin, month_cos
    """
    df = df.copy()
    hour  = df.index.hour
    month = df.index.month
    df["hour_sin"]  = np.sin(2 * np.pi * hour  / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * hour  / 24)
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)
    return df


print("[F1] FE_CONFIG defined.")
print(f"     Base vars   : {FE_CONFIG['base_vars']}")
print(f"     Lag vars    : {FE_CONFIG['lag_vars']}")
print(f"     Roll windows: {FE_CONFIG['roll_windows']}h")
print(f"     Lag offsets : {FE_CONFIG['lags']} hrs")
print("[F1] add_cyclical_time_features() defined.")


[F1] FE_CONFIG defined.
     Base vars   : ['temp', 'humidity', 'air_velocity', 'co2', 'solar_radiation', 'vpd', 'dew_point']
     Lag vars    : ['temp', 'humidity', 'vpd', 'air_velocity', 'solar_radiation', 'dew_point']
     Roll windows: [6, 12, 24]h
     Lag offsets : [1, 2, 3, 6, 12, 24] hrs
[F1] add_cyclical_time_features() defined.


In [58]:
# ─────────────────────────────────────────────────────────────────────────────
# F2 — compute_rolling_stats()
# Computes mean / max / min / std for every base variable over windows
# [6h, 12h, 24h].  min_periods=1 ensures no NaN for early rows.
# std is 0-filled for single-observation windows (avoids NaN → scaler issues).
# ─────────────────────────────────────────────────────────────────────────────

def compute_rolling_stats(df: pd.DataFrame, fe_config: dict) -> pd.DataFrame:
    """
    Append rolling mean/max/min/std columns for each base variable.

    Naming scheme: <var>_<stat>_<W>h
    E.g.: temp_mean_6h, humidity_std_24h, vpd_max_12h

    Returns
    -------
    df_out : pd.DataFrame with extra rolling stat columns appended.
    """
    df_out = df.copy()
    stats_added = []

    for var in fe_config["base_vars"]:
        if var not in df.columns:
            print(f"  [F2] WARNING: '{var}' not in df — skipped.")
            continue
        s = df[var]
        for W in fe_config["roll_windows"]:
            r = s.rolling(window=W, min_periods=1)
            df_out[f"{var}_mean_{W}h"] = r.mean()
            df_out[f"{var}_max_{W}h"]  = r.max()
            df_out[f"{var}_min_{W}h"]  = r.min()
            df_out[f"{var}_std_{W}h"]  = r.std().fillna(0.0)
            stats_added.extend([
                f"{var}_mean_{W}h", f"{var}_max_{W}h",
                f"{var}_min_{W}h",  f"{var}_std_{W}h",
            ])

    n_expected = len(fe_config["base_vars"]) * len(fe_config["roll_windows"]) * 4
    print(f"[F2] Rolling stats: {len(stats_added)} columns added "
          f"({n_expected} expected, {n_expected - len(stats_added)} skipped).")
    return df_out


print("[F2] compute_rolling_stats() defined.")


[F2] compute_rolling_stats() defined.


In [59]:
# ─────────────────────────────────────────────────────────────────────────────
# F3 — compute_lag_features()
# Lag features capture temporal persistence: conditions 1h, 2h, 3h, 6h,
# 12h, 24h ago inform the current risk trajectory.
#
# NaN Strategy:
#   - shift() creates NaN in the first max(lags)=24 rows.
#   - bfill() then ffill() fills these without introducing future leakage
#     (backfill only propagates values within the early warm-up period).
#   - The first window_N rows will be discarded anyway during RNN windowing.
# ─────────────────────────────────────────────────────────────────────────────

def compute_lag_features(df: pd.DataFrame, fe_config: dict) -> pd.DataFrame:
    """
    Append lag columns <var>_lag<L> for each (var, lag) pair.

    Naming scheme: <var>_lag<L>
    E.g.: temp_lag1, vpd_lag24, solar_radiation_lag6

    Returns
    -------
    df_out : pd.DataFrame with lag columns appended.
    """
    df_out = df.copy()
    lag_cols_added = []

    for var in fe_config["lag_vars"]:
        if var not in df.columns:
            print(f"  [F3] WARNING: '{var}' not in df — skipped.")
            continue
        for lag in fe_config["lags"]:
            col_name = f"{var}_lag{lag}"
            df_out[col_name] = df[var].shift(lag)
            lag_cols_added.append(col_name)

    # Fill NaN produced by shifting (only affects first max_lag rows)
    if lag_cols_added:
        df_out[lag_cols_added] = df_out[lag_cols_added].bfill().ffill()

    n_expected = len(fe_config["lag_vars"]) * len(fe_config["lags"])
    print(f"[F3] Lag features: {len(lag_cols_added)} columns added "
          f"({n_expected} expected). NaN filled via bfill→ffill.")
    return df_out


print("[F3] compute_lag_features() defined.")


[F3] compute_lag_features() defined.


In [60]:
# ─────────────────────────────────────────────────────────────────────────────
# F4 — compute_interaction_terms()
# Domain-informed multiplicative / additive feature pairs that capture
# second-order effects between environmental variables.
#
#   temp_x_rh      = T × RH  — heat-humidity load; high values → late_blight / leaf_mold
#   vpd_x_rh       = VPD × RH — paradoxical stress indicator; meaningful at extreme ends
#   dewpoint_spread = T − Td  — proxy for relative RH:
#                               ≈0 → near saturation (condensation risk)
#                               >5 → low RH (spider_mites / powdery_mildew risk)
# ─────────────────────────────────────────────────────────────────────────────

def compute_interaction_terms(df: pd.DataFrame) -> pd.DataFrame:
    """
    Append three interaction feature columns.

    Columns added:
        temp_x_rh      : temperature × humidity
        vpd_x_rh       : vpd × humidity
        dewpoint_spread : temperature − dew_point

    Returns
    -------
    df_out : pd.DataFrame with interaction columns appended.
    """
    df_out = df.copy()

    df_out["temp_x_rh"]       = df["temp"]     * df["humidity"]
    df_out["vpd_x_rh"]        = df["vpd"]      * df["humidity"]
    df_out["dewpoint_spread"]  = df["temp"]     - df["dew_point"]

    print("[F4] Interaction terms added: temp_x_rh, vpd_x_rh, dewpoint_spread")
    return df_out


print("[F4] compute_interaction_terms() defined.")


[F4] compute_interaction_terms() defined.


In [61]:
# ─────────────────────────────────────────────────────────────────────────────
# F5 — compute_day_night_rolling()
# Night-only rolling statistics for each base variable.
#
# Rationale: several pathogens (late_blight, leaf_mold) are strongly driven
# by overnight humidity and temperature dynamics. Daytime values dilute that
# signal in general rolling windows. Masking daytime to NaN and rolling
# over only night values gives the model a "night climate" channel.
#
# Strategy:
#   1. Mask each base variable: val if night (day_night_flag == 0) else NaN
#   2. Rolling mean over W hours with min_periods=1 (ignores NaN cells)
#   3. ffill().bfill() to fill daytime-gap NaNs (carry last night value forward)
# ─────────────────────────────────────────────────────────────────────────────

def compute_day_night_rolling(df: pd.DataFrame, fe_config: dict) -> pd.DataFrame:
    """
    Append night-only rolling mean columns for each base variable.

    Columns added: <var>_night_mean_<W>h  (W in roll_windows)

    Returns
    -------
    df_out : pd.DataFrame with night-segmented rolling columns appended.
    """
    df_out = df.copy()
    is_night = (df["day_night_flag"] == 0)   # True = night (flag == 0)
    added = []

    for var in fe_config["base_vars"]:
        if var not in df.columns:
            print(f"  [F5] WARNING: '{var}' not in df — skipped.")
            continue
        # Mask to night-only values; daytime hours become NaN
        night_series = df[var].where(is_night)
        for W in fe_config["roll_windows"]:
            col = f"{var}_night_mean_{W}h"
            rolled = (
                night_series
                .rolling(window=W, min_periods=1)
                .mean()
                .ffill()   # carry last night-window mean into daytime hours
                .bfill()   # handle series start (no prior night value)
            )
            df_out[col] = rolled
            added.append(col)

    n_expected = len(fe_config["base_vars"]) * len(fe_config["roll_windows"])
    print(f"[F5] Night-segmented rolling means: {len(added)} columns added "
          f"({n_expected} expected).")
    return df_out


print("[F5] compute_day_night_rolling() defined.")


[F5] compute_day_night_rolling() defined.


In [62]:
# ─────────────────────────────────────────────────────────────────────────────
# F6 — build_feature_matrix()
# Orchestrator: applies all FE transforms in sequence and assembles a
# stable-order feature matrix X and target matrix Y.
#
# Column order in X (deterministic, reproducible):
#   (a) Base sensor cols + day_night_flag       — raw environment
#   (b) Cyclical time cols                      — temporal encoding
#   (c) Exposure counter cols (from Section D)  — disease-specific accumulations
#   (d) Rolling stat cols (sorted)              — multi-scale sensor dynamics
#   (e) Lag feature cols (sorted)               — temporal persistence
#   (f) Interaction term cols                   — second-order effects
#   (g) Night-segmented rolling cols (sorted)   — overnight climate channel
#
# Y: 5 continuous risk columns (float 0–100) in CONFIG disease order.
# ─────────────────────────────────────────────────────────────────────────────

def build_feature_matrix(df_enriched: pd.DataFrame, fe_config: dict):
    """
    Apply all FE transforms and assemble (X, Y) matrices.

    Parameters
    ----------
    df_enriched : pd.DataFrame — output of run_disease_risk_pipeline() (Section D).
    fe_config   : dict         — FE_CONFIG defined in F1.

    Returns
    -------
    X               : pd.DataFrame — feature matrix, shape (n, F)
    Y               : pd.DataFrame — target matrix,  shape (n, 5)
    feature_columns : list[str]    — stable ordered feature column names
    target_columns  : list[str]    — ['risk_late_blight', ..., 'risk_spider_mites']
    """
    print("[F6] Building feature matrix — applying FE transforms in sequence …")
    _t_f6 = _time_mod.time()

    # ── 1. Cyclical time ─────────────────────────────────────────────────────
    df_fe = add_cyclical_time_features(df_enriched)

    # ── 2. Rolling stats ─────────────────────────────────────────────────────
    df_fe = compute_rolling_stats(df_fe, fe_config)

    # ── 3. Lag features ──────────────────────────────────────────────────────
    df_fe = compute_lag_features(df_fe, fe_config)

    # ── 4. Interaction terms ─────────────────────────────────────────────────
    df_fe = compute_interaction_terms(df_fe)

    # ── 5. Night-segmented rolling ───────────────────────────────────────────
    df_fe = compute_day_night_rolling(df_fe, fe_config)

    # ── 6. Identify column buckets ───────────────────────────────────────────
    # (a) Base sensors
    base_sensor_cols = [
        "temp", "humidity", "air_velocity", "co2",
        "solar_radiation", "vpd", "dew_point", "day_night_flag",
    ]
    base_sensor_cols = [c for c in base_sensor_cols if c in df_fe.columns]

    # (b) Cyclical time
    cyclical_cols = ["hour_sin", "hour_cos", "month_sin", "month_cos"]
    cyclical_cols = [c for c in cyclical_cols if c in df_fe.columns]

    # (c) Exposure counters from Section D (cond_, exposure_count_, night_exposure_)
    exposure_cols = sorted([
        c for c in df_fe.columns
        if (c.startswith("cond_")
            or c.startswith("exposure_count_")
            or c.startswith("night_exposure_"))
    ])

    # (d) Rolling stats (added in F2)
    base_var_set = set(fe_config["base_vars"])
    rolling_stats_cols = sorted([
        c for c in df_fe.columns
        if any(
            c.startswith(f"{v}_mean_") or c.startswith(f"{v}_max_")
            or c.startswith(f"{v}_min_") or c.startswith(f"{v}_std_")
            for v in base_var_set
        )
    ])

    # (e) Lag features (added in F3)
    lag_cols = sorted([c for c in df_fe.columns if "_lag" in c])

    # (f) Interaction terms (added in F4)
    interaction_cols = [
        c for c in ["temp_x_rh", "vpd_x_rh", "dewpoint_spread"]
        if c in df_fe.columns
    ]

    # (g) Night rolling (added in F5)
    night_rolling_cols = sorted([c for c in df_fe.columns if "_night_mean_" in c])

    # ── 7. Assemble stable-order feature list ─────────────────────────────────
    feature_columns = (
        base_sensor_cols
        + cyclical_cols
        + exposure_cols
        + rolling_stats_cols
        + lag_cols
        + interaction_cols
        + night_rolling_cols
    )
    # Safety: deduplicate while preserving order
    seen = set()
    feature_columns = [c for c in feature_columns
                       if not (c in seen or seen.add(c))]

    # ── 8. Target columns ─────────────────────────────────────────────────────
    target_columns = [f"risk_{d}" for d in fe_config["diseases"]]
    missing_targets = [c for c in target_columns if c not in df_fe.columns]
    if missing_targets:
        raise ValueError(f"[F6] Target columns missing from df_enriched: {missing_targets}. "
                         "Ensure Section D ran successfully.")

    # ── 9. Extract X and Y ───────────────────────────────────────────────────
    X = df_fe[feature_columns].copy()
    Y = df_fe[target_columns].copy()

    # ── 10. Final report ──────────────────────────────────────────────────────
    elapsed = _time_mod.time() - _t_f6
    print(f"\n[F6] ✓ Feature matrix assembled in {elapsed:.1f}s")
    print(f"     X shape : {X.shape}  ({len(feature_columns)} features × {len(X):,} rows)")
    print(f"     Y shape : {Y.shape}")
    print(f"\n     Feature budget by layer:")
    print(f"       (a) Base sensors     : {len(base_sensor_cols):>4}")
    print(f"       (b) Cyclical time    : {len(cyclical_cols):>4}")
    print(f"       (c) Exposure counters: {len(exposure_cols):>4}")
    print(f"       (d) Rolling stats    : {len(rolling_stats_cols):>4}")
    print(f"       (e) Lag features     : {len(lag_cols):>4}")
    print(f"       (f) Interaction terms: {len(interaction_cols):>4}")
    print(f"       (g) Night rolling    : {len(night_rolling_cols):>4}")
    print(f"       ─────────────────────────────")
    print(f"       Total               : {len(feature_columns):>4}")

    nan_x = X.isnull().sum().sum()
    nan_y = Y.isnull().sum().sum()
    if nan_x > 0:
        print(f"\n  ⚠ NaN in X: {nan_x} (check bfill/ffill in lag/night-rolling steps)")
    else:
        print(f"\n  ✓ No NaN in X")
    if nan_y > 0:
        print(f"  ⚠ NaN in Y: {nan_y}")
    else:
        print(f"  ✓ No NaN in Y")

    return X, Y, feature_columns, target_columns


print("[F6] build_feature_matrix() defined.")


[F6] build_feature_matrix() defined.


In [63]:
# ─────────────────────────────────────────────────────────────────────────────
# F7 — Run feature engineering pipeline
# Produces globals: X, Y, feature_columns, target_columns
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION F: FEATURE ENGINEERING")
print("=" * 70)

X, Y, feature_columns, target_columns = build_feature_matrix(df_enriched, FE_CONFIG)

# Quick peek
print("\n[F7] X column preview (first 10 / last 10):")
print(f"     First 10: {feature_columns[:10]}")
print(f"     Last  10: {feature_columns[-10:]}")
print(f"\n[F7] Y columns: {target_columns}")
print(f"\n[F7] Y statistics:")
print(Y.describe().round(2).to_string())


SECTION F: FEATURE ENGINEERING
[F6] Building feature matrix — applying FE transforms in sequence …
[F2] Rolling stats: 84 columns added (84 expected, 0 skipped).
[F3] Lag features: 36 columns added (36 expected). NaN filled via bfill→ffill.
[F4] Interaction terms added: temp_x_rh, vpd_x_rh, dewpoint_spread
[F5] Night-segmented rolling means: 21 columns added (21 expected).

[F6] ✓ Feature matrix assembled in 0.4s
     X shape : (17544, 181)  (181 features × 17,544 rows)
     Y shape : (17544, 5)

     Feature budget by layer:
       (a) Base sensors     :    8
       (b) Cyclical time    :    4
       (c) Exposure counters:   25
       (d) Rolling stats    :   84
       (e) Lag features     :   36
       (f) Interaction terms:    3
       (g) Night rolling    :   21
       ─────────────────────────────
       Total               :  181

  ✓ No NaN in X
  ✓ No NaN in Y

[F7] X column preview (first 10 / last 10):
     First 10: ['temp', 'humidity', 'air_velocity', 'co2', 'solar_radiatio

## Section G — Supervised Dataset Building (Time-Aware)

Transforms `X` and `Y` into model-ready supervised learning datasets for two
architectures — **Random Forest** (tabular) and **LSTM/GRU** (sequential).

### Design choices

#### G.1 — Random Forest datasets
**Choice: one multi-output RF dataset per forecast horizon H.**
- For each H in `CONFIG['horizon_H']` = `[6, 12, 24, 48]`, the target at row `t` is
  shifted by `−H` rows: $\hat{y}_t = Y_{t+H}$.
- Each dataset pairs the full instantaneous feature vector $\mathbf{x}_t$ with the
  future risk vector $\mathbf{y}_{t+H}$ (5 diseases, 5 columns → MultiOutputRegressor).
- Last `H` rows are dropped per dataset (NaN targets after shift).
- **Why per-horizon, not a single model?** Horizon-specific models learn separate
  temporal dynamics (e.g., 6h ahead vs 48h ahead) and enable horizon-resolved
  feature importances.

#### G.2 — RNN windowed datasets  
**Mode 1 (default) — Direct multi-horizon prediction:**
$$X^{\text{rnn}}[i] = X[t-N:t] \;\in \mathbb{R}^{N \times F}, \quad
  y^{\text{rnn}}[i] = \{Y[t+H]\}_{H \in \text{horizon\_H}} \;\in \mathbb{R}^{|H| \times 5}$$
- Each sample: last `N=24` hourly feature vectors → predicts risk levels at each
  horizon simultaneously (multi-head output).
- **Mode 2 (seq2seq, optional flag):** target is $Y[t:t+H_{\max}]$ — not implemented
  here; set `rnn_mode=2` to stub the data builder for future use.

Split boundary rules (no leakage):
- **Train windows** — window end `t` falls within the training date range.
- **Val/Test windows** — same rule applied to validation and test date ranges.
- Window features may reach back into a prior split start (this is unavoidable for
  rolling inputs and does NOT constitute leakage, as future targets are not visible).


In [64]:
# ─────────────────────────────────────────────────────────────────────────────
# G1 — build_rf_targets()
# For each forecast horizon H, shift the target matrix Y backward by H rows so
# that row t of (X, Y_H) means:
#   features at time t → predict risk at time t + H
#
# Dropped rows: last H rows per dataset (they would have NaN shifted targets).
# Output structure:
#   rf_datasets[H] = {
#       'X_train': ..., 'Y_train': ...,
#       'X_val':   ..., 'Y_val':   ...,
#       'X_test':  ..., 'Y_test':  ...,
#   }
# ─────────────────────────────────────────────────────────────────────────────

def build_rf_targets(X: pd.DataFrame, Y: pd.DataFrame,
                     df_splits: dict, fe_config: dict) -> dict:
    """
    Build one tabular RF dataset per forecast horizon H.

    Parameters
    ----------
    X           : feature matrix (DatetimeIndex, shape n × F)
    Y           : target matrix  (DatetimeIndex, shape n × 5)
    df_splits   : {'train': df, 'val': df, 'test': df} from Section C
    fe_config   : must contain 'horizon_H'

    Returns
    -------
    rf_datasets : dict keyed by horizon H (int), each value is a dict with
                  'X_train', 'Y_train', 'X_val', 'Y_val', 'X_test', 'Y_test'
                  as pd.DataFrames.
    """
    print("\n[G1] Building RF tabular datasets (one per horizon H) …")

    # Date bounds from splits for time-aware slicing
    train_end_ts = df_splits["train"].index.max()
    val_end_ts   = df_splits["val"].index.max()

    rf_datasets = {}

    for H in fe_config["horizon_H"]:
        # Shift Y by −H so that future target aligns with current features
        Y_H = Y.shift(-H)       # last H rows become NaN

        # Aligned feature + target (drop rows where target is NaN = last H rows)
        valid_mask = Y_H.notna().all(axis=1)
        X_H = X[valid_mask]
        Y_H = Y_H[valid_mask]

        # Time-aware split using DatetimeIndex
        X_tr = X_H[X_H.index <= train_end_ts]
        Y_tr = Y_H[Y_H.index <= train_end_ts]

        X_va = X_H[(X_H.index > train_end_ts) & (X_H.index <= val_end_ts)]
        Y_va = Y_H[(Y_H.index > train_end_ts) & (Y_H.index <= val_end_ts)]

        X_te = X_H[X_H.index > val_end_ts]
        Y_te = Y_H[Y_H.index > val_end_ts]

        rf_datasets[H] = {
            "X_train": X_tr, "Y_train": Y_tr,
            "X_val":   X_va, "Y_val":   Y_va,
            "X_test":  X_te, "Y_test":  Y_te,
        }

        print(f"     H={H:>3}h  →  "
              f"train={len(X_tr):>5,}  val={len(X_va):>5,}  test={len(X_te):>5,}  "
              f"(dropped last {H} rows for target shift)")

    print(f"\n[G1] RF datasets built for horizons: {list(rf_datasets.keys())} hours")
    return rf_datasets


print("[G1] build_rf_targets() defined.")


[G1] build_rf_targets() defined.


In [65]:
# ─────────────────────────────────────────────────────────────────────────────
# G2 — build_rnn_windows()
# Creates sliding-window arrays for LSTM/GRU training.
#
# Mode 1 (default, rnn_mode=1): Direct horizon prediction
#   For each valid position t (window end):
#     X_window = X[t-N : t]                → shape (N, F)
#     Y_target = Y[t + H - 1] for each H   → shape (|H|, 5)
#   (H=6 means "predict 6 hours after the window end")
#
# Mode 2 (stub, rnn_mode=2): Sequence-to-sequence
#   Y_target = Y[t : t + H_max]            → shape (H_max, 5)
#   Not fully trained in this notebook — builder is implemented but commented.
#
# Split logic:
#   Window assignment is determined by where the window "end" timestamp t falls:
#   - t ∈ train range → train window
#   - t ∈ val range   → val window
#   - t ∈ test range  → test window (only valid if t + H_max is within dataset)
# ─────────────────────────────────────────────────────────────────────────────

def build_rnn_windows(X: pd.DataFrame, Y: pd.DataFrame,
                      df_splits: dict, fe_config: dict,
                      rnn_mode: int = 1) -> dict:
    """
    Build sliding-window 3D arrays for LSTM/GRU input.

    Parameters
    ----------
    X         : feature matrix (DatetimeIndex, shape n × F)
    Y         : target matrix  (DatetimeIndex, shape n × 5)
    df_splits : {'train': df, 'val': df, 'test': df}
    fe_config : must contain 'window_N' and 'horizon_H'
    rnn_mode  : 1 = direct horizon prediction (default)
                2 = seq2seq stub (not trained here)

    Returns
    -------
    rnn_data : dict with keys 'train', 'val', 'test', each mapping to:
        'X' : np.ndarray shape (n_samples, N, F)
        'Y' : np.ndarray shape (n_samples, len(horizon_H), n_diseases)   [mode 1]
              or shape (n_samples, H_max, n_diseases)                     [mode 2]
        'timestamps': list of window-end timestamps (for traceability)
    metadata : dict (window_N, horizon_H, rnn_mode, n_features, n_diseases)
    """
    N          = fe_config["window_N"]       # lookback: 24 hours
    horizon_H  = fe_config["horizon_H"]      # e.g. [6, 12, 24, 48]
    H_max      = max(horizon_H)

    X_arr = X.values.astype(np.float32)   # (n, F)
    Y_arr = Y.values.astype(np.float32)   # (n, 5)
    ts    = X.index                        # DatetimeIndex for split assignment

    n_total    = len(X_arr)
    n_features = X_arr.shape[1]
    n_diseases = Y_arr.shape[1]

    # Date boundaries for split assignment
    train_end_ts = df_splits["train"].index.max()
    val_end_ts   = df_splits["val"].index.max()

    # Collect windows per split
    splits_X    = {"train": [], "val": [], "test": []}
    splits_Y    = {"train": [], "val": [], "test": []}
    splits_ts   = {"train": [], "val": [], "test": []}

    # Valid window positions: t must satisfy t >= N and t + H_max - 1 < n_total
    valid_end = n_total - H_max          # last valid window end (exclusive)

    print(f"\n[G2] Sliding window construction …")
    print(f"     N={N}h  horizons={horizon_H}  mode={rnn_mode}")
    print(f"     Valid positions: t ∈ [{N}, {valid_end}) = {valid_end - N:,} windows")

    for t in range(N, valid_end):
        # Determine split assignment by window-end timestamp
        t_ts = ts[t]
        if t_ts <= train_end_ts:
            split = "train"
        elif t_ts <= val_end_ts:
            split = "val"
        else:
            split = "test"

        # ── Input window X[t-N : t] ─────────────────────────────────────────
        x_win = X_arr[t - N : t]   # shape (N, F)

        # ── Target ──────────────────────────────────────────────────────────
        if rnn_mode == 1:
            # Direct: one target slice per horizon H
            # y_win[h] = Y at time t + H  (0-indexed offsets)
            y_win = np.stack([Y_arr[t + H - 1] for H in horizon_H], axis=0)
            # shape: (len(horizon_H), n_diseases)
        else:
            # Mode 2 stub: seq2seq — use Y[t : t + H_max]
            y_win = Y_arr[t : t + H_max]   # shape (H_max, n_diseases)

        splits_X[split].append(x_win)
        splits_Y[split].append(y_win)
        splits_ts[split].append(t_ts)

    # Convert lists → numpy arrays
    rnn_data = {}
    for split in ("train", "val", "test"):
        if splits_X[split]:
            rnn_data[split] = {
                "X":          np.array(splits_X[split],  dtype=np.float32),
                "Y":          np.array(splits_Y[split],  dtype=np.float32),
                "timestamps": splits_ts[split],
            }
        else:
            rnn_data[split] = {"X": np.empty((0, N, n_features)),
                               "Y": np.empty((0, len(horizon_H), n_diseases)),
                               "timestamps": []}
        n_s = len(splits_X[split])
        x_s = rnn_data[split]["X"].shape
        y_s = rnn_data[split]["Y"].shape
        print(f"     {split:<6}: {n_s:>6,} windows  X{x_s}  Y{y_s}")

    metadata = {
        "window_N":   N,
        "horizon_H":  horizon_H,
        "rnn_mode":   rnn_mode,
        "n_features": n_features,
        "n_diseases": n_diseases,
    }
    print(f"\n[G2] RNN windows built  (mode={rnn_mode}: "
          f"{'direct horizon' if rnn_mode == 1 else 'seq2seq'})")
    return rnn_data, metadata


print("[G2] build_rnn_windows() defined.")


[G2] build_rnn_windows() defined.


In [66]:
# ─────────────────────────────────────────────────────────────────────────────
# G3 — assemble_supervised_datasets()
# Orchestrates G1 + G2 into a single call.
# Returns all datasets packaged for downstream use.
# ─────────────────────────────────────────────────────────────────────────────

def assemble_supervised_datasets(X: pd.DataFrame, Y: pd.DataFrame,
                                  df_splits: dict, fe_config: dict,
                                  rnn_mode: int = 1) -> dict:
    """
    Run both RF and RNN dataset builders and return results in one dict.

    Returns
    -------
    datasets : {
        'rf':  rf_datasets  (from G1: horizon → train/val/test DataFrames),
        'rnn': rnn_data     (from G2: split → {'X', 'Y', 'timestamps'}),
        'rnn_metadata': metadata dict,
    }
    """
    print("=" * 70)
    print("SECTION G: SUPERVISED DATASET BUILDING")
    print("=" * 70)

    _t_g = _time_mod.time()

    # ── G1: RF tabular datasets ───────────────────────────────────────────────
    rf_datasets = build_rf_targets(X, Y, df_splits, fe_config)

    # ── G2: RNN windowed datasets ─────────────────────────────────────────────
    rnn_data, rnn_metadata = build_rnn_windows(X, Y, df_splits, fe_config,
                                               rnn_mode=rnn_mode)

    elapsed = _time_mod.time() - _t_g
    print(f"\n[G3] Supervised datasets assembled in {elapsed:.1f}s")

    # Summary table
    print(f"\n[G3] RF dataset summary ({len(fe_config['horizon_H'])} horizons):")
    print(f"     {'H':>5}  {'Train':>8}  {'Val':>8}  {'Test':>8}  {'Features':>9}  {'Targets':>8}")
    print(f"     {'─' * 60}")
    for H in fe_config["horizon_H"]:
        d = rf_datasets[H]
        print(f"     {H:>4}h  "
              f"{len(d['X_train']):>8,}  "
              f"{len(d['X_val']):>8,}  "
              f"{len(d['X_test']):>8,}  "
              f"{d['X_train'].shape[1]:>9}  "
              f"{d['Y_train'].shape[1]:>8}")

    print(f"\n[G3] RNN dataset summary (mode {rnn_metadata['rnn_mode']}):")
    for split in ("train", "val", "test"):
        x_shape = rnn_data[split]["X"].shape
        y_shape = rnn_data[split]["Y"].shape
        print(f"     {split:<6}: X{x_shape}  Y{y_shape}")

    return {
        "rf":           rf_datasets,
        "rnn":          rnn_data,
        "rnn_metadata": rnn_metadata,
    }


print("[G3] assemble_supervised_datasets() defined.")


[G3] assemble_supervised_datasets() defined.


In [67]:
# ─────────────────────────────────────────────────────────────────────────────
# G4 — Run supervised dataset assembly
# Globals produced: datasets (contains both 'rf' and 'rnn' sub-dicts)
# ─────────────────────────────────────────────────────────────────────────────
datasets = assemble_supervised_datasets(
    X, Y,
    df_splits,
    FE_CONFIG,
    rnn_mode=1,   # Mode 1: direct horizon prediction (multi-head targets)
)

# Convenience aliases for downstream cells
rf_datasets  = datasets["rf"]
rnn_data     = datasets["rnn"]
rnn_metadata = datasets["rnn_metadata"]


SECTION G: SUPERVISED DATASET BUILDING

[G1] Building RF tabular datasets (one per horizon H) …
     H=  6h  →  train=7,320  val=1,464  test=8,754  (dropped last 6 rows for target shift)
     H= 12h  →  train=7,320  val=1,464  test=8,748  (dropped last 12 rows for target shift)
     H= 24h  →  train=7,320  val=1,464  test=8,736  (dropped last 24 rows for target shift)
     H= 48h  →  train=7,320  val=1,464  test=8,712  (dropped last 48 rows for target shift)

[G1] RF datasets built for horizons: [6, 12, 24, 48] hours

[G2] Sliding window construction …
     N=24h  horizons=[6, 12, 24, 48]  mode=1
     Valid positions: t ∈ [24, 17496) = 17,472 windows
     train :  7,296 windows  X(7296, 24, 181)  Y(7296, 4, 5)
     val   :  1,464 windows  X(1464, 24, 181)  Y(1464, 4, 5)
     test  :  8,712 windows  X(8712, 24, 181)  Y(8712, 4, 5)

[G2] RNN windows built  (mode=1: direct horizon)

[G3] Supervised datasets assembled in 1.2s

[G3] RF dataset summary (4 horizons):
         H     Train     

## Section H — Scaling (No Leakage) + Feature Artifact Saving

### Scaler strategy
A **`StandardScaler`** (zero-mean, unit-variance) is fit **exclusively on `X_train`** from
the `H=6` RF dataset (smallest horizon, largest train set), then applied identically
to all validation and test splits — **no data from val/test ever informs the scaler**.

The same scaler is reused for RNN inputs: the RNN windows are constructed from the
*same scaled X matrix*, so no re-scaling is needed for the RNN arrays.

### Artifacts saved to `src/agritwin_gh/models/artifacts/dp_<run_id>/` (flat, prefixed)

| File | Prefix | Description |
|------|--------|-------------|
| `scaler.pkl` | shared | Fitted `StandardScaler` (joblib) |
| `feature_schema.json` | shared | Schema: column names, layer breakdown, window/lag/horizon params |
| `engineered_features_head.csv` | shared | First 50 rows of X (sanity check) |
| `engineered_features_tail.csv` | shared | Last 50 rows of X (sanity check) |
| `rf_feature_importance_<disease>.png` | `rf_` | (Section I) per-disease RF feature importance plot |
| `rnn_training_history.png` | `rnn_` | (Section J) LSTM/GRU loss curve |

> Model files are saved **flat** in `src/agritwin_gh/models/`:

> - `dp_rf_<disease>_<run_id>.joblib` — one RF model per disease (Section I)

> - `dp_rnn_<run_id>.keras` — LSTM/GRU model (Section J)> Sections I onward (RF training + LSTM) load from these files.

> After this section, all artifacts needed for model training are on disk.

In [68]:
# ─────────────────────────────────────────────────────────────────────────────
# H1 — fit_and_apply_scaler()
# StandardScaler fit on X_train (H=6 RF dataset, largest train partition).
# Returns:
#   X_scaled   : full scaled feature matrix (DatetimeIndex preserved)
#   scaler     : fitted StandardScaler object
#   rf_scaled  : scaled copies of all RF datasets
#   rnn_scaled : scaled numpy arrays for RNN splits
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

def fit_and_apply_scaler(X: pd.DataFrame, Y: pd.DataFrame,
                          rf_datasets: dict, rnn_data: dict,
                          fe_config: dict):
    """
    Fit StandardScaler on training data ONLY, then transform all splits.

    Scaler is fit on X_train from the smallest horizon (H=6) RF dataset.

    Parameters
    ----------
    X          : full feature matrix (n, F)
    Y          : full target matrix  (n, 5)
    rf_datasets: RF split dict keyed by horizon H
    rnn_data   : RNN windowed dict keyed by split name
    fe_config  : FE_CONFIG (for horizon list)

    Returns
    -------
    X_scaled       : pd.DataFrame — full scaled X (same DatetimeIndex)
    scaler         : fitted StandardScaler
    rf_scaled      : dict — scaled RF datasets {H: {'X_train':…, 'Y_train':…, …}}
    rnn_X_scaled   : dict — scaled RNN X arrays {'train':…, 'val':…, 'test':…}
    """
    print("\n[H1] Fitting StandardScaler on X_train (H=6 RF dataset) …")

    # ── 1. Fit scaler on train rows from H=6 RF dataset ───────────────────────
    H_anchor = fe_config["horizon_H"][0]    # smallest H (e.g. 6)
    X_train_rf = rf_datasets[H_anchor]["X_train"]

    scaler = StandardScaler()
    scaler.fit(X_train_rf.values)

    print(f"     Scaler fit on H={H_anchor}h train set  ({len(X_train_rf):,} rows)")
    print(f"     Feature means (first 5): {scaler.mean_[:5].round(4)}")
    print(f"     Feature stds  (first 5): {scaler.scale_[:5].round(4)}")

    # ── 2. Scale full X matrix ────────────────────────────────────────────────
    X_scaled_arr = scaler.transform(X.values).astype(np.float32)
    X_scaled = pd.DataFrame(X_scaled_arr, index=X.index, columns=X.columns)

    # ── 3. Scale all RF split DataFrames ─────────────────────────────────────
    rf_scaled = {}
    for H in fe_config["horizon_H"]:
        rf_scaled[H] = {}
        for split_key in ("train", "val", "test"):
            X_s = rf_datasets[H][f"X_{split_key}"]
            Y_s = rf_datasets[H][f"Y_{split_key}"]
            X_s_scaled = pd.DataFrame(
                scaler.transform(X_s.values).astype(np.float32),
                index=X_s.index, columns=X_s.columns
            )
            rf_scaled[H][f"X_{split_key}"] = X_s_scaled
            rf_scaled[H][f"Y_{split_key}"] = Y_s     # targets are NOT scaled

    # ── 4. Scale RNN X windows ────────────────────────────────────────────────
    # Each RNN X window has shape (n_samples, N, F); reshape to (n·N, F), scale, reshape back
    rnn_X_scaled = {}
    for split in ("train", "val", "test"):
        X_win = rnn_data[split]["X"]          # (n, N, F)
        if X_win.shape[0] == 0:
            rnn_X_scaled[split] = X_win
            continue
        n, N_w, F = X_win.shape
        X_reshaped = X_win.reshape(-1, F)
        X_reshaped_scaled = scaler.transform(X_reshaped).astype(np.float32)
        rnn_X_scaled[split] = X_reshaped_scaled.reshape(n, N_w, F)

    # ── 5. Sanity check: train mean ≈ 0 ──────────────────────────────────────
    train_mean = rf_scaled[H_anchor]["X_train"].mean().mean()
    print(f"\n[H1] Sanity — scaled train mean (should ≈ 0): {train_mean:.6f}")

    print(f"[H1] ✓ Scaler applied to all RF splits and RNN windows.")
    return X_scaled, scaler, rf_scaled, rnn_X_scaled


print("[H1] fit_and_apply_scaler() defined.")


[H1] fit_and_apply_scaler() defined.


In [69]:
# ─────────────────────────────────────────────────────────────────────────────
# H2 — save_feature_artifacts()
# Saves all Section F–H artifacts to P_ARTIFACTS_DP.
#
# Files saved:
#   scaler.pkl                   — fitted StandardScaler (joblib)
#   feature_schema.json          — column names, layer budget, params
#   engineered_features_head.csv — first 50 rows of X_scaled
#   engineered_features_tail.csv — last 50 rows of X_scaled
# ─────────────────────────────────────────────────────────────────────────────
import joblib

def save_feature_artifacts(X_scaled: pd.DataFrame,
                            scaler,
                            feature_columns: list,
                            target_columns: list,
                            fe_config: dict,
                            config: dict,
                            rf_datasets: dict,
                            rnn_metadata: dict) -> dict:
    """
    Persist all feature-engineering and scaling artifacts.

    Returns
    -------
    saved_paths : dict mapping artifact name → absolute path string.
    """
    run_id = config["run_id"]

    # ── 1. Scaler ─────────────────────────────────────────────────────────────
    p_scaler = P_ARTIFACTS_DP / "scaler.pkl"
    joblib.dump(scaler, p_scaler)
    print(f"[H2] Saved scaler          : {p_scaler}")

    # ── 2. Feature schema JSON ────────────────────────────────────────────────
    # Categorise columns into their engineering layers (mirrors F6 logic)
    base_sensor_cols = [
        "temp", "humidity", "air_velocity", "co2",
        "solar_radiation", "vpd", "dew_point", "day_night_flag",
    ]
    cyclical_cols    = ["hour_sin", "hour_cos", "month_sin", "month_cos"]
    exposure_cols    = [c for c in feature_columns
                        if (c.startswith("cond_") or c.startswith("exposure_count_")
                            or c.startswith("night_exposure_"))]
    rolling_cols     = [c for c in feature_columns
                        if any(c.startswith(f"{v}_mean_") or c.startswith(f"{v}_max_")
                               or c.startswith(f"{v}_min_")  or c.startswith(f"{v}_std_")
                               for v in fe_config["base_vars"])]
    lag_cols         = [c for c in feature_columns if "_lag" in c]
    interaction_cols = [c for c in feature_columns
                        if c in {"temp_x_rh", "vpd_x_rh", "dewpoint_spread"}]
    night_roll_cols  = [c for c in feature_columns if "_night_mean_" in c]

    schema = {
        "run_id":           run_id,
        "n_features":       len(feature_columns),
        "feature_columns":  feature_columns,          # stable ordered list
        "target_columns":   target_columns,
        "layers": {
            "base_sensors":     base_sensor_cols,
            "cyclical_time":    cyclical_cols,
            "exposure_counters":exposure_cols,
            "rolling_stats":    rolling_cols,
            "lag_features":     lag_cols,
            "interaction_terms":interaction_cols,
            "night_rolling":    night_roll_cols,
        },
        "parameters": {
            "base_vars":     fe_config["base_vars"],
            "lag_vars":      fe_config["lag_vars"],
            "roll_windows_h":fe_config["roll_windows"],
            "lags":          fe_config["lags"],
            "window_N":      fe_config["window_N"],
            "horizon_H":     fe_config["horizon_H"],
        },
        "scaler":       "sklearn.preprocessing.StandardScaler",
        "scaler_file":  "scaler.pkl",
        "rnn_mode":     rnn_metadata["rnn_mode"],
        "n_diseases":   rnn_metadata["n_diseases"],
    }
    p_schema = P_ARTIFACTS_DP / "feature_schema.json"
    with open(p_schema, "w") as f:
        json.dump(schema, f, indent=2)
    print(f"[H2] Saved feature schema  : {p_schema}")

    # ── 3. Head / tail CSVs of scaled X ──────────────────────────────────────
    p_head = P_ARTIFACTS_DP / "engineered_features_head.csv"
    p_tail = P_ARTIFACTS_DP / "engineered_features_tail.csv"

    X_scaled.head(50).to_csv(p_head)
    X_scaled.tail(50).to_csv(p_tail)
    print(f"[H2] Saved feature head CSV: {p_head}  (50 rows)")
    print(f"[H2] Saved feature tail CSV: {p_tail}  (50 rows)")

    return {
        "scaler":                  str(p_scaler),
        "feature_schema":          str(p_schema),
        "engineered_features_head":str(p_head),
        "engineered_features_tail":str(p_tail),
    }


print("[H2] save_feature_artifacts() defined.")


[H2] save_feature_artifacts() defined.


In [70]:
# ─────────────────────────────────────────────────────────────────────────────
# H3 — Run scaling + artifact saving (final cell for Sections F–H)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION H: SCALING + FEATURE ARTIFACT SAVING")
print("=" * 70)

_t_h = _time_mod.time()

# ── 1. Fit scaler and apply to all splits ────────────────────────────────────
X_scaled, scaler, rf_scaled, rnn_X_scaled = fit_and_apply_scaler(
    X, Y, rf_datasets, rnn_data, FE_CONFIG
)

# ── 2. Save all feature artifacts ────────────────────────────────────────────
feat_artifact_paths = save_feature_artifacts(
    X_scaled, scaler,
    feature_columns, target_columns,
    FE_CONFIG, CONFIG,
    rf_scaled, rnn_metadata,
)

# ── 3. Final summary ─────────────────────────────────────────────────────────
total_elapsed = _time_mod.time() - _t0_total

print(f"\n{'=' * 70}")
print(f"Pipeline Sections A–H complete.  Total elapsed: {total_elapsed:.1f}s")
print(f"Run ID : {CONFIG['run_id']}")
print(f"{'=' * 70}")
print(f"\n[H3] Artifact index (this run):")

# Combine all artifact paths from Sections E and H
all_artifacts = {**saved_paths, **feat_artifact_paths}
for k, v in all_artifacts.items():
    print(f"     {k:<35} → {v}")

print(f"\n[H3] Scaled dataset shapes:")
print(f"     X_scaled (full)        : {X_scaled.shape}")
for H in FE_CONFIG["horizon_H"]:
    tr_sh = rf_scaled[H]["X_train"].shape
    va_sh = rf_scaled[H]["X_val"].shape
    te_sh = rf_scaled[H]["X_test"].shape
    print(f"     RF H={H:>3}h  train={tr_sh}  val={va_sh}  test={te_sh}")
for split in ("train", "val", "test"):
    x_sh = rnn_X_scaled[split].shape
    y_sh = rnn_data[split]["Y"].shape
    print(f"     RNN {split:<6}  X={x_sh}  Y={y_sh}")

print(f"\n{'=' * 70}")
print(f"Next: Section I — RandomForest baseline training + feature importance")
print(f"      Section J — LSTM/GRU multi-step risk forecasting")
print(f"{'=' * 70}")


SECTION H: SCALING + FEATURE ARTIFACT SAVING

[H1] Fitting StandardScaler on X_train (H=6 RF dataset) …
     Scaler fit on H=6h train set  (7,320 rows)
     Feature means (first 5): [ 31.0778  69.8272   2.0545 417.5329  70.7907]
     Feature stds  (first 5): [ 3.7236 10.4075  0.5288 22.3993 91.1269]

[H1] Sanity — scaled train mean (should ≈ 0): 0.000000
[H1] ✓ Scaler applied to all RF splits and RNN windows.
[H2] Saved scaler          : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\scaler.pkl
[H2] Saved feature schema  : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\feature_schema.json
[H2] Saved feature head CSV: E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\engineered_features_head.csv  (50 rows)
[H2] Saved feature tail CSV: E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\dp_20260305_111754\engineered_features_tail.csv  (50 rows)

Pipeline Sections A–H complete.  Total elapsed: 2213.8s
Run ID : dp_20260305_111754

[H3] A

## Section I — Random Forest Training: Baseline + Feature Importance

### Modeling Design Choice

**Choice: one `MultiOutputRegressor(RandomForestRegressor)` per forecast horizon H — 4 models total.**

| Option | Models | Interpretability | Chosen? |
|--------|--------|------------------|---------|
| A — multi-output RF per horizon | `len(horizon_H)` = 4 | Per-disease importances via `estimators_[i]` | ✅ |
| B — RF per disease per horizon | 5 × 4 = 20 | Cleanest isolation | Too many for baseline |
| C — one RF for all diseases & horizons | 1 | Least flexible | Not chosen |

**Why A:**  
- 4 models reflect the 4 temporal horizons (6h, 12h, 24h, 48h) — each horizon has genuinely different predictive dynamics.  
- `MultiOutputRegressor` trains one RF per target column internally, so disease-level feature importances are still accessible via `model.estimators_[i].feature_importances_`.  
- 4 `.joblib` files on disk: `dp_rf_H{H}_{run_id}.joblib`.

**Evaluation metrics:**
- **Regression:** MAE, RMSE per disease per horizon (val + test)
- **Classification (High risk detection):** Binarise predictions at the label boundary (≥67 = High) and report Precision / Recall / F1 for the "High" class per disease — this directly measures the early-warning capability.

**Plots saved to `P_ARTIFACTS_DP` (prefix `rf_`):**
- `rf_feature_importance_H{H}.png` — top-30 importances (averaged across 5-disease estimators)
- `rf_trajectory_H{H}.png` — true vs predicted risk, one test week, all 5 diseases


In [71]:
# ─────────────────────────────────────────────────────────────────────────────
# I1 — train_rf_models()
# Trains one MultiOutputRegressor(RandomForestRegressor) per horizon H.
# Input: scaled RF datasets produced by Sections G + H.
# Output: dict keyed by H → fitted MultiOutputRegressor.
#
# MultiOutputRegressor internally creates one RF per target column (disease),
# so `model.estimators_[i]` gives the per-disease RF for disease index i.
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
import time as _time_mod

def train_rf_models(rf_scaled: dict, config: dict) -> dict:
    """
    Train one MultiOutputRegressor per forecast horizon on the training split.

    Parameters
    ----------
    rf_scaled : dict — {H: {'X_train':…, 'Y_train':…, 'X_val':…, …}} from Section H
    config    : CONFIG dict (reads 'rf_params', 'seed', 'horizon_H', 'diseases')

    Returns
    -------
    rf_models : dict {H: fitted MultiOutputRegressor}
    """
    rf_p = config["rf_params"]
    diseases = config["diseases"]
    print("[I1] Training Random Forest models …")
    print(f"     Params: n_estimators={rf_p['n_estimators']}  "
          f"max_depth={rf_p['max_depth']}  "
          f"min_samples_leaf={rf_p['min_samples_leaf']}")
    print(f"     Horizons: {config['horizon_H']}h\n")

    rf_models = {}

    for H in config["horizon_H"]:
        X_tr = rf_scaled[H]["X_train"].values
        Y_tr = rf_scaled[H]["Y_train"].values    # (n_train, 5 diseases)

        base_rf = RandomForestRegressor(
            n_estimators    = rf_p["n_estimators"],
            max_depth       = rf_p["max_depth"],
            min_samples_leaf= rf_p["min_samples_leaf"],
            random_state    = rf_p["random_state"],
            n_jobs          = rf_p["n_jobs"],
        )
        model = MultiOutputRegressor(base_rf, n_jobs=1)

        t0 = _time_mod.time()
        model.fit(X_tr, Y_tr)
        elapsed = _time_mod.time() - t0

        rf_models[H] = model
        print(f"     H={H:>3}h → trained in {elapsed:>5.1f}s  "
              f"(n_train={len(X_tr):,}  n_features={X_tr.shape[1]}  n_targets=5)")

    print(f"\n[I1] All RF models trained.")
    return rf_models


print("[I1] train_rf_models() defined.")


[I1] train_rf_models() defined.


In [72]:
# ─────────────────────────────────────────────────────────────────────────────
# I2 — evaluate_rf()
# Computes regression metrics (MAE, RMSE) and High-risk detection metrics
# (Precision, Recall, F1) for every disease × horizon × split combination.
#
# High-risk binarisation:
#   predicted high if pred_risk >= CONFIG['risk_label_bins']['high'][0]  (= 67)
#   true high      if true_risk  >= same threshold
# This directly measures the early-warning capability — the most safety-critical
# dimension for a disease progression system.
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    precision_score, recall_score, f1_score,
)
import numpy as np

def evaluate_rf(rf_models: dict, rf_scaled: dict, config: dict) -> dict:
    """
    Evaluate all RF models on val and test splits.

    Returns
    -------
    metrics : nested dict
      metrics[H][split][disease] = {
          'MAE': float, 'RMSE': float,
          'Precision_High': float, 'Recall_High': float, 'F1_High': float
      }
    Also prints a formatted summary table.
    """
    diseases      = config["diseases"]
    high_thresh   = config["risk_label_bins"]["high"][0]   # 67.0
    metrics       = {}

    print("[I2] RF Evaluation — MAE / RMSE / High-class Precision-Recall-F1")
    print(f"     High-risk threshold: ≥ {high_thresh}")

    for H in config["horizon_H"]:
        model    = rf_models[H]
        metrics[H] = {}

        for split in ("val", "test"):
            X_s = rf_scaled[H][f"X_{split}"].values
            Y_s = rf_scaled[H][f"Y_{split}"].values    # (n, 5)

            Y_pred = model.predict(X_s)                 # (n, 5)
            Y_pred = np.clip(Y_pred, 0.0, 100.0)

            metrics[H][split] = {}
            for d_idx, disease in enumerate(diseases):
                y_true = Y_s[:, d_idx]
                y_hat  = Y_pred[:, d_idx]

                mae  = mean_absolute_error(y_true, y_hat)
                rmse = np.sqrt(mean_squared_error(y_true, y_hat))

                # High-risk binary detection
                y_true_bin = (y_true >= high_thresh).astype(int)
                y_hat_bin  = (y_hat  >= high_thresh).astype(int)

                # zero_division=0 avoids warnings when class absent in split
                prec = precision_score(y_true_bin, y_hat_bin, zero_division=0)
                rec  = recall_score(   y_true_bin, y_hat_bin, zero_division=0)
                f1   = f1_score(       y_true_bin, y_hat_bin, zero_division=0)

                metrics[H][split][disease] = {
                    "MAE":           round(float(mae),  3),
                    "RMSE":          round(float(rmse), 3),
                    "Precision_High":round(float(prec), 3),
                    "Recall_High":   round(float(rec),  3),
                    "F1_High":       round(float(f1),   3),
                }

        # ── Pretty-print table for this horizon ──────────────────────────────
        print(f"\n  H = {H}h")
        print(f"  {'Disease':<22} {'Split':<6} {'MAE':>7} {'RMSE':>7} "
              f"{'P_High':>8} {'R_High':>8} {'F1_High':>8}")
        print(f"  {'─' * 75}")
        for disease in diseases:
            for split in ("val", "test"):
                m = metrics[H][split][disease]
                print(f"  {disease:<22} {split:<6} "
                      f"{m['MAE']:>7.3f} {m['RMSE']:>7.3f} "
                      f"{m['Precision_High']:>8.3f} {m['Recall_High']:>8.3f} "
                      f"{m['F1_High']:>8.3f}")

    print(f"\n[I2] Evaluation complete.")
    return metrics


print("[I2] evaluate_rf() defined.")


[I2] evaluate_rf() defined.


In [73]:
# ─────────────────────────────────────────────────────────────────────────────
# I3 — plot_rf_feature_importance()
# Produces one bar plot per horizon H showing the top-30 most important
# features, averaged across the 5 per-disease estimators.
#
# Averaging rationale: because all 5 estimators share the same feature space,
# averaging their importances gives a single "how predictive is this feature
# for disease risk in general" view — useful for feature selection later.
# Per-disease importances are saved to the metrics dict for inspection.
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")       # non-interactive backend for saving to disk

def plot_rf_feature_importance(rf_models: dict, feature_columns: list,
                                config: dict, top_n: int = 30) -> dict:
    """
    Plot and save top-N feature importances (averaged across diseases) for each H.

    Also embeds per-disease importances in the returned dict for logging.

    Parameters
    ----------
    rf_models       : {H: MultiOutputRegressor}
    feature_columns : ordered list of feature names (from Section F)
    config          : CONFIG dict
    top_n           : number of top features to display

    Returns
    -------
    importance_data : {H: {'mean': np.array, 'per_disease': {disease: np.array}}}
    saved_plots     : {H: str path}
    """
    diseases     = config["diseases"]
    saved_plots  = {}
    importance_data = {}

    for H in config["horizon_H"]:
        model = rf_models[H]

        # Collect per-disease importances from underlying estimators
        per_disease_imp = {}
        imp_matrix = np.zeros((len(diseases), len(feature_columns)))
        for d_idx, disease in enumerate(diseases):
            imp = model.estimators_[d_idx].feature_importances_
            per_disease_imp[disease] = imp
            imp_matrix[d_idx] = imp

        # Average across diseases
        mean_imp = imp_matrix.mean(axis=0)
        importance_data[H] = {
            "mean":       mean_imp,
            "per_disease":per_disease_imp,
        }

        # ── Sort and select top_n ─────────────────────────────────────────────
        top_idx   = np.argsort(mean_imp)[::-1][:top_n]
        top_names = [feature_columns[i] for i in top_idx]
        top_vals  = mean_imp[top_idx]

        # ── Plot ─────────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.30)))
        colours = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, top_n))
        bars = ax.barh(range(top_n), top_vals[::-1], color=colours[::-1])

        ax.set_yticks(range(top_n))
        ax.set_yticklabels([n.replace("_", " ") for n in top_names[::-1]], fontsize=8)
        ax.set_xlabel("Mean Importance (averaged across 5 diseases)", fontsize=10)
        ax.set_title(
            f"RF Feature Importance — Top {top_n}  |  Horizon H={H}h  |  Run: {config['run_id']}",
            fontsize=10, pad=12
        )
        ax.grid(axis="x", alpha=0.3)

        # Annotate bars with value
        for bar_obj, val in zip(bars, top_vals[::-1]):
            ax.text(val + mean_imp.max() * 0.005, bar_obj.get_y() + bar_obj.get_height() / 2,
                    f"{val:.4f}", va="center", fontsize=7)

        plt.tight_layout()
        p_fig = P_ARTIFACTS_DP / f"rf_feature_importance_H{H}.png"
        fig.savefig(p_fig, dpi=120, bbox_inches="tight")
        plt.close(fig)
        saved_plots[H] = str(p_fig)
        print(f"[I3] H={H}h  top feature: '{top_names[0]}' ({top_vals[0]:.4f}) → {p_fig.name}")

    return importance_data, saved_plots


print("[I3] plot_rf_feature_importance() defined.")


[I3] plot_rf_feature_importance() defined.


In [74]:
# ─────────────────────────────────────────────────────────────────────────────
# I4 — plot_rf_trajectory()
# True vs Predicted risk trajectory for a selected 7-day window in the test set.
# One subplot per disease, all horizons overlaid.
#
# Strategy:
#   - Select the first full 7-day (168-hour) window in the test split.
#   - Overlay true risk and predicted risk for all 4 horizons on the same axes.
#   - This visualises how well each horizon anticipates true risk evolution.
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.dates as mdates

def plot_rf_trajectory(rf_models: dict, rf_scaled: dict,
                        config: dict, n_hours: int = 168) -> str:
    """
    Plot true vs predicted risk trajectories for a test-set window.

    Parameters
    ----------
    rf_models  : {H: MultiOutputRegressor}
    rf_scaled  : RF scaled datasets from Section H
    config     : CONFIG dict
    n_hours    : window length in hours (default 7 days = 168h)

    Returns
    -------
    p_fig : str — path to saved PNG
    """
    diseases  = config["diseases"]
    n_disease = len(diseases)
    H_list    = config["horizon_H"]
    COLOURS   = {6: "#e63946", 12: "#f4a261", 24: "#2a9d8f", 48: "#457b9d"}

    # Use H=24 test data for the index / true-value reference
    H_ref = 24
    X_te = rf_scaled[H_ref]["X_test"]
    Y_te = rf_scaled[H_ref]["Y_test"]

    if len(X_te) < n_hours:
        n_hours = len(X_te)
        print(f"  [I4] Test set shorter than requested window — using {n_hours}h.")

    # Slice first n_hours rows
    X_win = X_te.iloc[:n_hours]
    Y_win = Y_te.iloc[:n_hours].values       # (n_hours, 5) — true risk at t+H_ref
    ts    = X_win.index                      # DatetimeIndex for x-axis

    fig, axes = plt.subplots(n_disease, 1,
                             figsize=(16, 3.5 * n_disease),
                             sharex=True)

    for d_idx, (ax, disease) in enumerate(zip(axes, diseases)):
        # True risk (from H_ref dataset)
        ax.plot(ts, Y_win[:, d_idx],
                color="black", linewidth=1.5, label="True risk (t+24h)", zorder=5)

        # Predicted risk for each horizon
        for H in H_list:
            X_h  = rf_scaled[H]["X_test"].iloc[:n_hours]
            Y_h  = rf_scaled[H]["Y_test"].iloc[:n_hours].values
            Y_pred_h = np.clip(rf_models[H].predict(X_h.values), 0, 100)

            ax.plot(ts, Y_pred_h[:, d_idx],
                    color=COLOURS[H], linewidth=0.9, alpha=0.8,
                    linestyle="--", label=f"Pred H={H}h")

        # Risk-level threshold bands
        bins = config["risk_label_bins"]
        ax.axhspan(bins["high"][0], 100,    alpha=0.06, color="red")
        ax.axhspan(bins["medium"][0], bins["high"][0] - 1, alpha=0.05, color="orange")
        ax.axhline(bins["high"][0],   color="red",    linestyle=":", linewidth=0.9, alpha=0.7)
        ax.axhline(bins["medium"][0], color="orange", linestyle=":", linewidth=0.9, alpha=0.7)

        ax.set_ylim(0, 105)
        ax.set_ylabel("Risk (0–100)", fontsize=9)
        ax.set_title(disease.replace("_", " ").title(), fontsize=10, loc="left", pad=4)
        ax.grid(True, alpha=0.25)
        ax.legend(loc="upper right", fontsize=8, ncol=len(H_list) + 1)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    axes[-1].xaxis.set_major_locator(mdates.DayLocator())
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha="right")

    fig.suptitle(
        f"RF: True vs Predicted Risk — Test Set ({n_hours}h window)\n"
        f"Run: {config['run_id']}",
        fontsize=11, y=1.01
    )
    plt.tight_layout()

    p_fig = P_ARTIFACTS_DP / "rf_trajectory.png"
    fig.savefig(p_fig, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"[I4] Trajectory plot saved: {p_fig}")
    return str(p_fig)


print("[I4] plot_rf_trajectory() defined.")


[I4] plot_rf_trajectory() defined.


In [75]:
# ─────────────────────────────────────────────────────────────────────────────
# I5 — save_rf_artifacts()
# Saves:
#   P_MODELS_DIR / dp_rf_H{H}_{run_id}.joblib    ← one per horizon
#   P_ARTIFACTS_DP / rf_metrics.json              ← full metrics dict
#   P_ARTIFACTS_DP / rf_metrics_summary.csv       ← flat CSV for easy inspection
# ─────────────────────────────────────────────────────────────────────────────

def save_rf_artifacts(rf_models: dict, metrics: dict,
                      importance_plots: dict, traj_plot: str,
                      config: dict) -> dict:
    """
    Persist RF model files and evaluation artifacts.

    Returns
    -------
    saved : dict mapping label → absolute path string
    """
    import joblib, json, csv
    run_id   = config["run_id"]
    diseases = config["diseases"]
    saved    = {}

    # ── 1. Save model .joblib files (one per horizon) ─────────────────────────
    for H, model in rf_models.items():
        fname = f"dp_rf_H{H}_{run_id}.joblib"
        p_model = P_MODELS_DIR / fname
        joblib.dump(model, p_model)
        saved[f"rf_model_H{H}"] = str(p_model)
        print(f"[I5] Saved model  : {p_model}")

    # ── 2. rf_metrics.json ────────────────────────────────────────────────────
    p_metrics = P_ARTIFACTS_DP / "rf_metrics.json"
    with open(p_metrics, "w") as f:
        json.dump({"run_id": run_id, "metrics": metrics}, f, indent=2)
    saved["rf_metrics_json"] = str(p_metrics)
    print(f"[I5] Saved metrics : {p_metrics}")

    # ── 3. rf_metrics_summary.csv — flat table for easy copy-paste into report
    p_csv = P_ARTIFACTS_DP / "rf_metrics_summary.csv"
    rows  = []
    for H in config["horizon_H"]:
        for split in ("val", "test"):
            for disease in diseases:
                m = metrics[H][split][disease]
                rows.append({
                    "horizon_h": H,
                    "split":     split,
                    "disease":   disease,
                    **m,
                })
    if rows:
        with open(p_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
    saved["rf_metrics_csv"] = str(p_csv)
    print(f"[I5] Saved metrics CSV: {p_csv}")

    # ── 4. Register plot paths ─────────────────────────────────────────────────
    for H, p in importance_plots.items():
        saved[f"rf_importance_plot_H{H}"] = p
    saved["rf_trajectory_plot"] = traj_plot

    return saved


print("[I5] save_rf_artifacts() defined.")


[I5] save_rf_artifacts() defined.


In [76]:
# ─────────────────────────────────────────────────────────────────────────────
# I6 — Run RF pipeline: train → evaluate → plot → save
# All outputs are stored in globals rf_models, rf_metrics, rf_saved_paths.
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION I: RANDOM FOREST TRAINING + EVALUATION")
print("=" * 70)

_t_i = _time_mod.time()

# ── 1. Train ──────────────────────────────────────────────────────────────────
rf_models = train_rf_models(rf_scaled, CONFIG)

# ── 2. Evaluate ──────────────────────────────────────────────────────────────
rf_metrics = evaluate_rf(rf_models, rf_scaled, CONFIG)

# ── 3. Feature importance plots ──────────────────────────────────────────────
print("\n[I3] Plotting feature importances …")
importance_data, importance_plots = plot_rf_feature_importance(
    rf_models, feature_columns, CONFIG, top_n=30
)

# ── 4. Trajectory plot ────────────────────────────────────────────────────────
print("\n[I4] Plotting true vs predicted trajectory (test set, 7-day window) …")
traj_plot = plot_rf_trajectory(rf_models, rf_scaled, CONFIG, n_hours=168)

# ── 5. Save models + artifacts ────────────────────────────────────────────────
print("\n[I5] Saving RF models and artifacts …")
rf_saved_paths = save_rf_artifacts(
    rf_models, rf_metrics, importance_plots, traj_plot, CONFIG
)

# ── 6. Final summary ──────────────────────────────────────────────────────────
elapsed_i = _time_mod.time() - _t_i
total_elapsed = _time_mod.time() - _t0_total

print(f"\n{'=' * 70}")
print(f"Section I complete.  Elapsed: {elapsed_i:.1f}s  |  Total: {total_elapsed:.1f}s")
print(f"Run ID : {CONFIG['run_id']}")
print(f"{'=' * 70}")

print(f"\n[I6] Saved artifact index:")
for k, v in rf_saved_paths.items():
    print(f"     {k:<35} → {v}")

# ── 7. Quick per-disease F1–High summary for H=24 ────────────────────────────
print(f"\n[I6] High-risk F1 summary (H=24h, TEST split):")
print(f"     {'Disease':<22}  {'MAE':>7}  {'RMSE':>7}  {'F1_High':>9}")
print(f"     {'─' * 55}")
for disease in CONFIG["diseases"]:
    m = rf_metrics[24]["test"][disease]
    print(f"     {disease:<22}  {m['MAE']:>7.3f}  {m['RMSE']:>7.3f}  {m['F1_High']:>9.3f}")

print(f"\n{'=' * 70}")
print(f"Next: Section J — LSTM/GRU multi-step risk forecasting")
print(f"{'=' * 70}")


SECTION I: RANDOM FOREST TRAINING + EVALUATION
[I1] Training Random Forest models …
     Params: n_estimators=300  max_depth=12  min_samples_leaf=5
     Horizons: [6, 12, 24, 48]h

     H=  6h → trained in 250.2s  (n_train=7,320  n_features=181  n_targets=5)
     H= 12h → trained in 283.9s  (n_train=7,320  n_features=181  n_targets=5)
     H= 24h → trained in 287.5s  (n_train=7,320  n_features=181  n_targets=5)
     H= 48h → trained in 302.5s  (n_train=7,320  n_features=181  n_targets=5)

[I1] All RF models trained.
[I2] RF Evaluation — MAE / RMSE / High-class Precision-Recall-F1
     High-risk threshold: ≥ 67

  H = 6h
  Disease                Split      MAE    RMSE   P_High   R_High  F1_High
  ───────────────────────────────────────────────────────────────────────────
  late_blight            val      6.324  11.335    0.975    0.893    0.932
  late_blight            test     1.578   5.947    0.983    0.873    0.925
  leaf_mold              val      6.853  12.171    0.932    0.971    

## Section J — LSTM/GRU Multi-Step Disease Risk Forecasting

Sequence-to-multi-output model that predicts **5 disease risk indices (0–100)** at all
configured horizons simultaneously using a shared LSTM encoder with per-horizon output heads.

### Design choices
| Decision | Choice | Rationale |
|---|---|---|
| Cell type | **LSTM** | Stable gradient flow for 24-step greenhouse sequences |
| GRU note | Acceptable alternative | ~15 % faster; negligible accuracy difference at N=24 |
| Output heads | One `Dense(5)` head per H | Avoids target mixing; clean per-horizon metrics |
| Loss | MAE (config-switchable to Huber) | Huber dampens outliers; MAE interpretable in risk-index units |
| Normalization | LayerNorm after each LSTM (config) | Stabilises activations without batch-size dependence |
| Dropout | 0.2 / 0.1 per LSTM layer | Regularisation for ~10 k–50 k training samples |

### Callbacks
- `EarlyStopping` (patience=15, restore best weights)
- `ModelCheckpoint` → `dp_rnn_{run_id}.keras`
- `ReduceLROnPlateau` (factor=0.5, patience=7)
- `CSVLogger` → `rnn_training_history.csv`

### Artifact naming (flat `P_ARTIFACTS_DP`)
| File | Description |
|---|---|
| `rnn_training_history.csv` | Epoch-level loss/MAE per head |
| `rnn_metrics.json` | Val + test MAE/RMSE/P/R/F1-High per disease per horizon |
| `rnn_metrics_summary.csv` | Wide-format summary table |
| `rnn_trajectory_<disease>.png` | True vs predicted for all horizons (168 h window) |
| `rnn_horizon_error.png` | Mean MAE vs horizon H bar chart |
| `rnn_calibration.png` | Predicted vs true scatter (test, H=24) |


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# J0 — LSTM hyper-parameter block (patched into CONFIG at runtime)
# Adjust here; all downstream cells read from CONFIG["lstm"].
# ─────────────────────────────────────────────────────────────────────────────
CONFIG["lstm"] = {
    # Encoder architecture
    "units_1":       128,      # LSTM layer-1 hidden units
    "units_2":       64,       # LSTM layer-2 hidden units
    "dense_units":   64,       # Shared dense after encoder
    "dropout_1":     0.20,     # Dropout after LSTM-1
    "dropout_2":     0.10,     # Dropout after LSTM-2
    "layer_norm":    True,     # LayerNormalization after each LSTM

    # Compilation
    "lr":            1e-3,
    "loss":          "mae",    # "mae" | "huber" (TF HuberLoss with delta=5.0)

    # Training
    "epochs":        150,
    "batch_size":    64,

    # EarlyStopping
    "patience":      15,
    "min_delta":     1e-4,

    # ReduceLROnPlateau
    "reduce_lr_factor":   0.50,
    "reduce_lr_patience": 7,
    "min_lr":             1e-6,
}

# Output path for the Keras model (flat — saved directly in models folder)
P_LSTM_MODEL_DIR  = P_MODELS_DIR  # no subfolder
P_LSTM_MODEL_PATH = P_MODELS_DIR / f"lstm_{CONFIG['run_id']}.keras"

print("[J0] LSTM config patch applied.")
print(f"     Model will be saved to : {P_LSTM_MODEL_PATH.relative_to(_ROOT)}")
print(f"     Artifacts directory    : {P_ARTIFACTS_DP.relative_to(_ROOT)}")
print(f"     Loss function          : {CONFIG['lstm']['loss']}")
print(f"     Epochs (max)           : {CONFIG['lstm']['epochs']}  "
      f"patience={CONFIG['lstm']['patience']}")


[J0] LSTM config patch applied.
     Model will be saved to : src\agritwin_gh\models\disease_progression\lstm_dp_20260305_111754.keras
     Artifacts directory    : src\agritwin_gh\models\artifacts\dp_20260305_111754
     Loss function          : mae
     Epochs (max)           : 150  patience=15


In [86]:

# ─────────────────────────────────────────────────────────────────────────────
# J1 — prepare_rnn_tensors()
# Converts scaled RNN input windows + target dicts into float32 NumPy tensors
# ready for Keras .fit().  No data leakage: scaler was fit on train only (H1).
#
# Note on rnn_data["Y"] layout (from G2, mode=1):
#   rnn_data[split]["Y"] has shape (n, len(horizon_H), n_diseases)
#   axis-1 is the horizon position index, NOT the horizon value itself.
#   e.g. horizon_H=[6,12,24,48] → Y[:, 0, :]=H6, Y[:, 1, :]=H12, …
# ─────────────────────────────────────────────────────────────────────────────

def prepare_rnn_tensors(rnn_X_scaled_dict, rnn_data_dict, config):
    """
    Build float32 tensors from scaled RNN inputs and target arrays.

    Parameters
    ----------
    rnn_X_scaled_dict : dict
        {split: ndarray (n, N, F)} — output of fit_and_apply_scaler() H1.
    rnn_data_dict : dict
        {split: {'X': ..., 'Y': ndarray (n, |H|, D)}} — from G2.
        Y axis-1 is the positional index into config['horizon_H'].
    config : dict
        CONFIG dict (uses 'horizon_H', 'diseases').

    Returns
    -------
    tensors : dict
        {split: {'X': float32 (n,N,F), 'Y': {H: float32 (n,D)}}}
    n_timesteps : int   (N)
    n_features  : int   (F)
    """
    splits = ["train", "val", "test"]
    tensors = {}

    for split in splits:
        X_raw = rnn_X_scaled_dict[split]
        X = np.asarray(X_raw, dtype=np.float32)          # (n, N, F)

        Y_all = rnn_data_dict[split]["Y"]                 # (n, |H|, D)

        Y_dict = {}
        for h_idx, H in enumerate(config["horizon_H"]):
            # Slice the correct horizon by positional index, not H value
            Y_dict[H] = np.asarray(Y_all[:, h_idx, :], dtype=np.float32)  # (n, D)

        tensors[split] = {"X": X, "Y": Y_dict}

    n_timesteps = tensors["train"]["X"].shape[1]
    n_features  = tensors["train"]["X"].shape[2]
    n_diseases  = len(config["diseases"])

    # ── Sanity checks ─────────────────────────────────────────────────────────
    for split in splits:
        X_s = tensors[split]["X"]
        assert X_s.ndim == 3, f"[J1] {split} X must be 3-D, got {X_s.ndim}"
        assert X_s.shape[1] == n_timesteps, f"[J1] Timestep mismatch in {split}"
        assert X_s.shape[2] == n_features,  f"[J1] Feature mismatch in {split}"
        for H in config["horizon_H"]:
            Y_s = tensors[split]["Y"][H]
            assert Y_s.shape == (X_s.shape[0], n_diseases), \
                f"[J1] Y shape mismatch in {split}, H={H}: {Y_s.shape}"

    print("[J1] RNN tensors ready (no NaN check — scaler handles this):")
    print(f"     Timesteps (N) : {n_timesteps}")
    print(f"     Features  (F) : {n_features}")
    print(f"     Diseases  (D) : {n_diseases}")
    print(f"     Horizons  (H) : {config['horizon_H']}")
    for split in splits:
        n_s = tensors[split]["X"].shape[0]
        print(f"     [{split:5s}]  n={n_s:6d}   "
              f"X: {tensors[split]['X'].shape}   "
              f"Y[H=6]: {tensors[split]['Y'][6].shape}")

    return tensors, n_timesteps, n_features


In [87]:
# ─────────────────────────────────────────────────────────────────────────────
# J2 — build_lstm_model()
# Multi-head LSTM: shared encoder → one Dense(D) head per horizon H.
# GRU is equally valid (faster ~15 %) for N=24 short sequences; LSTM chosen
# for established gradient stability on multivariate greenhouse sensor data.
# ─────────────────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_lstm_model(n_timesteps, n_features, n_diseases, horizon_H, config):
    """
    Build and compile the multi-head LSTM model.

    Architecture
    ────────────
    Input (N, F)
      └─ LSTM(units_1, return_seq=True) → [LayerNorm] → Dropout
          └─ LSTM(units_2, return_seq=False) → [LayerNorm] → Dropout
              └─ Dense(dense_units, relu)  ← shared encoder output
                  ├─ Head H6  : Dense(32, relu) → Dense(D) → Clip[0,100]
                  ├─ Head H12 : Dense(32, relu) → Dense(D) → Clip[0,100]
                  ├─ Head H24 : Dense(32, relu) → Dense(D) → Clip[0,100]
                  └─ Head H48 : Dense(32, relu) → Dense(D) → Clip[0,100]

    Parameters
    ----------
    n_timesteps : int   N  (lookback window length)
    n_features  : int   F  (number of input features)
    n_diseases  : int   D  (number of disease risk outputs)
    horizon_H   : list  e.g. [6, 12, 24, 48]
    config      : dict  CONFIG dict (reads CONFIG['lstm'])

    Returns
    -------
    keras.Model
    """
    mc = config["lstm"]

    inp = keras.Input(shape=(n_timesteps, n_features), name="input_seq")

    # ── Shared LSTM encoder ───────────────────────────────────────────────────
    x = layers.LSTM(mc["units_1"], return_sequences=True, name="lstm_1")(inp)
    if mc.get("layer_norm", False):
        x = layers.LayerNormalization(name="ln_1")(x)
    x = layers.Dropout(mc["dropout_1"], name="drop_1")(x)

    x = layers.LSTM(mc["units_2"], return_sequences=False, name="lstm_2")(x)
    if mc.get("layer_norm", False):
        x = layers.LayerNormalization(name="ln_2")(x)
    x = layers.Dropout(mc["dropout_2"], name="drop_2")(x)

    x = layers.Dense(mc["dense_units"], activation="relu", name="shared_dense")(x)

    # ── Per-horizon output heads ──────────────────────────────────────────────
    outputs = {}
    for H in horizon_H:
        h = layers.Dense(32, activation="relu", name=f"head_H{H}")(x)
        out = layers.Dense(n_diseases, name=f"output_H{H}")(h)
        # Clip predictions to valid risk range [0, 100]
        out = layers.Lambda(
            lambda t: tf.clip_by_value(t, 0.0, 100.0),
            name=f"clip_H{H}"
        )(out)
        outputs[f"output_H{H}"] = out

    model = keras.Model(inputs=inp, outputs=outputs, name="dp_lstm")

    # ── Compile ───────────────────────────────────────────────────────────────
    loss_fn = mc.get("loss", "mae")
    if loss_fn == "huber":
        _loss_obj = keras.losses.Huber(delta=5.0)
        losses_dict = {f"output_H{H}": _loss_obj for H in horizon_H}
    else:
        losses_dict = {f"output_H{H}": "mae" for H in horizon_H}

    # Equal weight per horizon head
    loss_weights = {f"output_H{H}": 1.0 for H in horizon_H}

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=mc["lr"]),
        loss=losses_dict,
        loss_weights=loss_weights,
        metrics={f"output_H{H}": [keras.metrics.MeanAbsoluteError(name="mae")]
                 for H in horizon_H},
    )

    print("[J2] Model built successfully.")
    model.summary(line_length=90, print_fn=lambda s: print("  " + s))
    return model


In [88]:
# ─────────────────────────────────────────────────────────────────────────────
# J3 — train_lstm_model()
# Trains the multi-head LSTM with four Keras callbacks:
#   1. EarlyStopping          → restores best weights
#   2. ModelCheckpoint        → saves best model to .keras file
#   3. ReduceLROnPlateau      → halves LR on plateau
#   4. CSVLogger              → epoch-level loss log
# ─────────────────────────────────────────────────────────────────────────────

def train_lstm_model(model, tensors, config, model_path, history_csv_path):
    """
    Fit the LSTM model on training tensors; validate on val split.

    Parameters
    ----------
    model            : compiled keras.Model from J2
    tensors          : dict {split: {'X', 'Y': {H: array}}} from J1
    config           : CONFIG dict
    model_path       : pathlib.Path  — where to save best model (.keras)
    history_csv_path : pathlib.Path  — where to write CSVLogger output

    Returns
    -------
    history : keras.callbacks.History
    """
    mc = config["lstm"]

    # ── Format inputs/outputs for Keras ──────────────────────────────────────
    def _pack_Y(split):
        return {f"output_H{H}": tensors[split]["Y"][H]
                for H in config["horizon_H"]}

    X_train = tensors["train"]["X"]
    Y_train = _pack_Y("train")
    X_val   = tensors["val"]["X"]
    Y_val   = _pack_Y("val")

    # Monitor the average val loss across all heads
    monitor_metric = "val_loss"

    # ── Callbacks ─────────────────────────────────────────────────────────────
    cb_early = keras.callbacks.EarlyStopping(
        monitor=monitor_metric,
        patience=mc["patience"],
        min_delta=mc["min_delta"],
        restore_best_weights=True,
        verbose=1,
    )

    cb_ckpt = keras.callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor=monitor_metric,
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    )

    cb_reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor=monitor_metric,
        factor=mc["reduce_lr_factor"],
        patience=mc["reduce_lr_patience"],
        min_lr=mc["min_lr"],
        verbose=1,
    )

    cb_csv = keras.callbacks.CSVLogger(
        filename=str(history_csv_path),
        separator=",",
        append=False,
    )

    callbacks = [cb_early, cb_ckpt, cb_reduce_lr, cb_csv]

    print(f"[J3] Training LSTM — max {mc['epochs']} epochs, "
          f"batch_size={mc['batch_size']}, patience={mc['patience']}")
    print(f"     Train samples : {X_train.shape[0]}")
    print(f"     Val   samples : {X_val.shape[0]}")
    print(f"     Best model → {model_path.name}")
    print(f"     History CSV → {history_csv_path.name}")

    _t = _time_mod.time()
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=mc["epochs"],
        batch_size=mc["batch_size"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed = _time_mod.time() - _t

    best_epoch = int(np.argmin(history.history["val_loss"])) + 1
    best_val   = min(history.history["val_loss"])
    print(f"\n[J3] Training complete in {elapsed:.1f}s")
    print(f"     Best epoch    : {best_epoch}")
    print(f"     Best val_loss : {best_val:.4f}")

    return history


In [89]:
# ─────────────────────────────────────────────────────────────────────────────
# J4 — evaluate_lstm()
# Regression : MAE, RMSE per disease × horizon × split (val + test)
# High-class  : Precision / Recall / F1  (risk >= 67 → HIGH)
# Returns nested dict: metrics[H][split][disease] = {MAE, RMSE, …}
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_lstm(model, tensors, config):
    """
    Evaluate the LSTM model on val and test splits.

    Parameters
    ----------
    model   : trained keras.Model
    tensors : dict from J1
    config  : CONFIG dict

    Returns
    -------
    metrics : dict  [H][split][disease] → {MAE, RMSE, Precision_High,
                                            Recall_High, F1_High}
    """
    from sklearn.metrics import precision_recall_fscore_support

    HIGH_THRESHOLD = CONFIG["risk_label_bins"]["high"][0]   # 67
    diseases       = config["diseases"]
    horizon_H      = config["horizon_H"]
    eval_splits    = ["val", "test"]

    metrics = {H: {s: {} for s in eval_splits} for H in horizon_H}

    for split in eval_splits:
        X = tensors[split]["X"]                          # (n, N, F)
        preds_dict = model.predict(X, verbose=0)         # {output_HH: (n, D)}

        for H in horizon_H:
            Y_true = tensors[split]["Y"][H]              # (n, D)
            Y_pred = preds_dict[f"output_H{H}"]         # (n, D)
            Y_pred = np.clip(Y_pred, 0.0, 100.0)

            for d_idx, disease in enumerate(diseases):
                y_t = Y_true[:, d_idx]
                y_p = Y_pred[:, d_idx]

                mae  = float(np.mean(np.abs(y_t - y_p)))
                rmse = float(np.sqrt(np.mean((y_t - y_p) ** 2)))

                # High-risk classification
                t_bin = (y_t >= HIGH_THRESHOLD).astype(int)
                p_bin = (y_p >= HIGH_THRESHOLD).astype(int)
                prec, rec, f1, _ = precision_recall_fscore_support(
                    t_bin, p_bin, pos_label=1,
                    average="binary", zero_division=0
                )
                metrics[H][split][disease] = {
                    "MAE":           mae,
                    "RMSE":          rmse,
                    "Precision_High": float(prec),
                    "Recall_High":    float(rec),
                    "F1_High":        float(f1),
                }

    # ── Pretty print H=24 test summary ────────────────────────────────────────
    print("\n[J4] LSTM evaluation — H=24h, TEST split:")
    print(f"     {'Disease':<22}  {'MAE':>7}  {'RMSE':>7}  "
          f"{'Prec_H':>7}  {'Rec_H':>7}  {'F1_H':>7}")
    print(f"     {'─' * 65}")
    for disease in diseases:
        m = metrics[24]["test"][disease]
        print(f"     {disease:<22}  {m['MAE']:>7.3f}  {m['RMSE']:>7.3f}  "
              f"{m['Precision_High']:>7.3f}  {m['Recall_High']:>7.3f}  "
              f"{m['F1_High']:>7.3f}")

    return metrics


In [90]:
# ─────────────────────────────────────────────────────────────────────────────
# J5 — plot_lstm_results()
# Generates three diagnostic plot sets saved to P_ARTIFACTS_DP:
#   A) rnn_trajectory_<disease>.png  — true vs predicted (168 h, all H)
#   B) rnn_horizon_error.png         — mean MAE vs horizon H (test split)
#   C) rnn_calibration.png           — pred vs true scatter (H=24, test)
# ─────────────────────────────────────────────────────────────────────────────

def plot_lstm_results(model, tensors, lstm_metrics, config, n_hours=168):
    """
    Generate and save all LSTM diagnostic plots.

    Returns
    -------
    saved_plots : dict  {plot_key: Path}
    """
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec

    diseases  = config["diseases"]
    horizon_H = config["horizon_H"]
    out_dir   = P_ARTIFACTS_DP

    HIGH_TH  = CONFIG["risk_label_bins"]["high"][0]    # 67
    MED_TH   = CONFIG["risk_label_bins"]["medium"][0]  # 33
    COLORS   = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]  # per horizon
    saved_plots = {}

    # ── Helper: run predictions on test set ───────────────────────────────────
    X_test = tensors["test"]["X"]
    preds  = model.predict(X_test, verbose=0)  # {output_HH: (n, D)}

    # ──────────────────────────────────────────────────────────────────────────
    # A) Trajectory plots — one figure per disease
    # ──────────────────────────────────────────────────────────────────────────
    for d_idx, disease in enumerate(diseases):
        fig, ax = plt.subplots(figsize=(14, 4))

        # True trajectory from H=6 (shortest horizon → densest coverage)
        y_true = tensors["test"]["Y"][6][:n_hours, d_idx]
        ax.plot(y_true, color="black", lw=1.8, label="True", zorder=5)

        for H, color in zip(horizon_H, COLORS):
            y_pred = np.clip(preds[f"output_H{H}"][:n_hours, d_idx], 0, 100)
            ax.plot(y_pred, color=color, lw=1.2, alpha=0.85,
                    label=f"LSTM H={H}h", linestyle="--")

        # Risk-band shading
        ax.axhspan(HIGH_TH, 100, alpha=0.08, color="red",    label="High ≥67")
        ax.axhspan(MED_TH,  HIGH_TH, alpha=0.06, color="orange",  label="Medium")
        ax.axhline(HIGH_TH, color="red",    lw=0.8, ls=":")
        ax.axhline(MED_TH,  color="orange", lw=0.8, ls=":")

        ax.set_title(f"Disease Risk Trajectory — {disease}  (test, first {n_hours}h)",
                    fontsize=12)
        ax.set_xlabel("Hour offset (test set)")
        ax.set_ylabel("Risk Index (0–100)")
        ax.set_ylim(-2, 105)
        ax.legend(loc="upper right", fontsize=8, ncol=3)
        ax.grid(True, alpha=0.3)

        slug = disease.replace(" ", "_").replace("/", "_")
        p = out_dir / f"rnn_trajectory_{slug}.png"
        fig.savefig(p, dpi=150, bbox_inches="tight")
        plt.close(fig)
        saved_plots[f"trajectory_{disease}"] = p
        print(f"  [J5-A] Saved: {p.name}")

    # ──────────────────────────────────────────────────────────────────────────
    # B) Horizon error plot — mean MAE per H (test, averaged over diseases)
    # ──────────────────────────────────────────────────────────────────────────
    mean_mae_per_H = []
    mean_rmse_per_H = []
    for H in horizon_H:
        maes  = [lstm_metrics[H]["test"][d]["MAE"]  for d in diseases]
        rmses = [lstm_metrics[H]["test"][d]["RMSE"] for d in diseases]
        mean_mae_per_H.append(np.mean(maes))
        mean_rmse_per_H.append(np.mean(rmses))

    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(horizon_H))
    w = 0.35
    ax.bar(x - w/2, mean_mae_per_H,  w, label="MAE",  color="#2196F3", alpha=0.85)
    ax.bar(x + w/2, mean_rmse_per_H, w, label="RMSE", color="#FF5722", alpha=0.85)
    for i, (m, r) in enumerate(zip(mean_mae_per_H, mean_rmse_per_H)):
        ax.text(i - w/2, m + 0.3, f"{m:.2f}", ha="center", va="bottom", fontsize=8)
        ax.text(i + w/2, r + 0.3, f"{r:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f"H={H}h" for H in horizon_H])
    ax.set_xlabel("Forecast Horizon")
    ax.set_ylabel("Error (risk index units)")
    ax.set_title("LSTM — Mean MAE & RMSE vs Forecast Horizon (test, avg over diseases)")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    p = out_dir / "rnn_horizon_error.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots["horizon_error"] = p
    print(f"  [J5-B] Saved: {p.name}")

    # ──────────────────────────────────────────────────────────────────────────
    # C) Calibration scatter — predicted vs true (H=24, test split)
    # ──────────────────────────────────────────────────────────────────────────
    n_diseases = len(diseases)
    ncols = min(3, n_diseases)
    nrows = (n_diseases + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows),
                             squeeze=False)

    Y_true_24 = tensors["test"]["Y"][24]
    Y_pred_24 = np.clip(preds["output_H24"], 0, 100)

    for d_idx, disease in enumerate(diseases):
        row, col = divmod(d_idx, ncols)
        ax = axes[row][col]
        ax.scatter(Y_true_24[:, d_idx], Y_pred_24[:, d_idx],
                   alpha=0.25, s=8, color="#5C6BC0")
        lo, hi = 0, 100
        ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="Perfect")
        ax.axhline(HIGH_TH, color="red",    lw=0.7, ls=":", alpha=0.6)
        ax.axvline(HIGH_TH, color="red",    lw=0.7, ls=":", alpha=0.6)
        ax.set_xlabel("True risk index")
        ax.set_ylabel("Predicted risk index")
        ax.set_title(f"{disease}")
        ax.set_xlim(-2, 105); ax.set_ylim(-2, 105)
        ax.legend(fontsize=7)

    # Hide unused subplots
    for idx in range(n_diseases, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row][col].set_visible(False)

    fig.suptitle("LSTM Calibration — Predicted vs True Risk (H=24h, Test)", fontsize=12)
    fig.tight_layout()
    p = out_dir / "rnn_calibration.png"
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    saved_plots["calibration"] = p
    print(f"  [J5-C] Saved: {p.name}")

    return saved_plots


In [91]:
# ─────────────────────────────────────────────────────────────────────────────
# J6 — save_lstm_artifacts()
# Saves LSTM model + all metrics + history + registered plot paths.
# The .keras model is saved by ModelCheckpoint; this function writes metrics.
# ─────────────────────────────────────────────────────────────────────────────

def save_lstm_artifacts(model, history, lstm_metrics, lstm_plots, config,
                        model_path, skip_save_model=True):
    """
    Save LSTM evaluation metrics and build the artifact path registry.

    Parameters
    ----------
    model           : keras.Model  (best weights already restored)
    history         : keras.callbacks.History
    lstm_metrics    : dict from J4
    lstm_plots      : dict from J5  {key: Path}
    config          : CONFIG dict
    model_path      : Path  — .keras file location
    skip_save_model : bool  — True if ModelCheckpoint already saved the model

    Returns
    -------
    saved_paths : dict  {artifact_key: str(path)}
    """
    import json, csv

    saved_paths = {}
    diseases  = config["diseases"]
    horizon_H = config["horizon_H"]
    run_id    = config["run_id"]

    # ── 1. Re-save model (optional override) ─────────────────────────────────
    if not skip_save_model:
        model.save(str(model_path))
        print(f"  [J6] LSTM model saved (override): {model_path.name}")
    else:
        if model_path.exists():
            print(f"  [J6] LSTM model present (from checkpoint): {model_path.name}")
        else:
            model.save(str(model_path))
            print(f"  [J6] LSTM model saved (checkpoint missing): {model_path.name}")
    saved_paths["lstm_model"] = str(model_path)

    # ── 2. metrics JSON ───────────────────────────────────────────────────────
    # Convert nested dict → JSON-serialisable
    def _to_jsonable(obj):
        if isinstance(obj, dict):
            return {str(k): _to_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, (np.floating, float)):
            return round(float(obj), 6)
        return obj

    metrics_json_path = P_ARTIFACTS_DP / "rnn_metrics.json"
    with open(metrics_json_path, "w") as f:
        json.dump(_to_jsonable(lstm_metrics), f, indent=2)
    saved_paths["rnn_metrics_json"] = str(metrics_json_path)
    print(f"  [J6] rnn_metrics.json saved.")

    # ── 3. metrics summary CSV ────────────────────────────────────────────────
    metrics_csv_path = P_ARTIFACTS_DP / "rnn_metrics_summary.csv"
    rows = []
    for H in horizon_H:
        for split in ["val", "test"]:
            for disease in diseases:
                m = lstm_metrics[H][split][disease]
                rows.append({
                    "run_id":  run_id, "horizon_H": H, "split": split,
                    "disease": disease,
                    **{k: round(v, 6) for k, v in m.items()},
                })
    with open(metrics_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    saved_paths["rnn_metrics_csv"] = str(metrics_csv_path)
    print(f"  [J6] rnn_metrics_summary.csv saved  ({len(rows)} rows).")

    # ── 4. Register plot paths ────────────────────────────────────────────────
    for key, p in lstm_plots.items():
        saved_paths[f"plot_{key}"] = str(p)

    # history CSV was written by CSVLogger; just register it
    hist_csv = P_ARTIFACTS_DP / "rnn_training_history.csv"
    if hist_csv.exists():
        saved_paths["rnn_history_csv"] = str(hist_csv)

    print(f"\n  [J6] LSTM artifact registry ({len(saved_paths)} items):")
    for k, v in saved_paths.items():
        print(f"       {k:<35} → {v}")

    return saved_paths


In [92]:
# ─────────────────────────────────────────────────────────────────────────────
# J7 — Run LSTM pipeline: prep → build → train → evaluate → plot → save
# Globals produced: lstm_model, lstm_tensors, lstm_metrics, lstm_saved_paths
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("SECTION J: LSTM/GRU MULTI-STEP DISEASE RISK FORECASTING")
print("=" * 70)

_t_j = _time_mod.time()

# ── J1: Tensor prep ───────────────────────────────────────────────────────────
print("\n[J1] Preparing RNN tensors …")
lstm_tensors, _N, _F = prepare_rnn_tensors(rnn_X_scaled, rnn_data, CONFIG)
_D = len(CONFIG["diseases"])

# ── J2: Build model ───────────────────────────────────────────────────────────
print("\n[J2] Building LSTM architecture …")
lstm_model = build_lstm_model(
    n_timesteps=_N,
    n_features=_F,
    n_diseases=_D,
    horizon_H=CONFIG["horizon_H"],
    config=CONFIG,
)

# ── J3: Train ─────────────────────────────────────────────────────────────────
print("\n[J3] Training …")
_hist_csv = P_ARTIFACTS_DP / "rnn_training_history.csv"
lstm_history = train_lstm_model(
    model=lstm_model,
    tensors=lstm_tensors,
    config=CONFIG,
    model_path=P_LSTM_MODEL_PATH,
    history_csv_path=_hist_csv,
)

# ── J4: Evaluate ──────────────────────────────────────────────────────────────
print("\n[J4] Evaluating …")
lstm_metrics = evaluate_lstm(lstm_model, lstm_tensors, CONFIG)

# ── J5: Plots ─────────────────────────────────────────────────────────────────
print("\n[J5] Generating diagnostic plots …")
lstm_plots = plot_lstm_results(
    lstm_model, lstm_tensors, lstm_metrics, CONFIG, n_hours=168
)

# ── J6: Save ──────────────────────────────────────────────────────────────────
print("\n[J6] Saving LSTM artifacts …")
lstm_saved_paths = save_lstm_artifacts(
    model=lstm_model,
    history=lstm_history,
    lstm_metrics=lstm_metrics,
    lstm_plots=lstm_plots,
    config=CONFIG,
    model_path=P_LSTM_MODEL_PATH,
    skip_save_model=True,    # ModelCheckpoint already saved the best model
)

# ── Summary ───────────────────────────────────────────────────────────────────
elapsed_j  = _time_mod.time() - _t_j
total_el   = _time_mod.time() - _t0_total

print(f"\n{'=' * 70}")
print(f"Section J complete.  Elapsed: {elapsed_j:.1f}s  |  Total: {total_el:.1f}s")
print(f"Epochs run : {len(lstm_history.history['val_loss'])}")
print(f"Best epoch : {int(np.argmin(lstm_history.history['val_loss'])) + 1}")
print(f"{'=' * 70}")
print("Next: Section K — High-risk reporting logic")
print(f"{'=' * 70}")


SECTION J: LSTM/GRU MULTI-STEP DISEASE RISK FORECASTING

[J1] Preparing RNN tensors …
[J1] RNN tensors ready (no NaN check — scaler handles this):
     Timesteps (N) : 24
     Features  (F) : 181
     Diseases  (D) : 5
     Horizons  (H) : [6, 12, 24, 48]
     [train]  n=  7296   X: (7296, 24, 181)   Y[H=6]: (7296, 5)
     [val  ]  n=  1464   X: (1464, 24, 181)   Y[H=6]: (1464, 5)
     [test ]  n=  8712   X: (8712, 24, 181)   Y[H=6]: (8712, 5)

[J2] Building LSTM architecture …

[J2] Model built successfully.


  Model: "dp_lstm"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)             ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_seq (InputLayer)   │ (None, 24, 181)      │            0 │ -                     │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ lstm_1 (LSTM)            │ (None, 24, 128)      │      158,720 │ input_seq[0][0]       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ ln_1                     │ (None, 24, 128)      │          256 │ lstm_1[0][0]          │
│ (LayerNormalization)     │                      │              │                       │
├──────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ drop_1 (Dropout)         │ (None, 24, 128)      │            0 │ ln_1

## Section K — High-Risk Disease Reporting Logic

Converts raw model predictions into a **human-readable, actionable alert** suitable
for greenhouse operators (IEEE-compatible terse format).

### Report rules
| Condition | Output |
|---|---|
| ≥ 1 disease HIGH | Top-2 HIGH diseases ranked by risk index |
| Only 1 disease HIGH | That disease only |
| 0 HIGH diseases | `"No high-risk disease forecast"` (no list) |

### Risk aggregation (configurable)
| Mode | Description |
|---|---|
| `"risk_at_horizon"` | Risk value at the single chosen horizon (e.g. H=24) |
| `"max_next_24h"` *(default)* | `max(H=6, H=12, H=24)` — worst-case in next 24 h |

### Reason extraction
Dominant factors are ranked by magnitude from the most recent feature row:
`exposure_24h_*` → `night_exposure_*` → `airflow_low` (proxy: low ventilation rate)
→ `radiation_high` (proxy: high PAR/pyranometer) → fallback temperature/RH.


In [93]:
# ─────────────────────────────────────────────────────────────────────────────
# K1 — extract_dominant_reason()
# Parses a single-row feature Series and returns a short, human-readable
# reason string by ranking the top contributing environmental signals.
# ─────────────────────────────────────────────────────────────────────────────

# Signal priority order (highest → lowest) with human labels
_SIGNAL_PRIORITY = [
    # (feature_substring, label_when_high, label_when_low, direction)
    # direction: "high" → alert when value is HIGH, "low" → alert when LOW
    ("exposure_24h_rh",          "High RH exposure (24h)",          None,                    "high"),
    ("exposure_24h_temp",        "High temp exposure (24h)",         None,                    "high"),
    ("exposure_24h_vpd",         "High VPD stress (24h)",            None,                    "high"),
    ("night_exposure_rh",        "High RH during night",             None,                    "high"),
    ("night_exposure_temp",      "Warm nights",                      None,                    "high"),
    ("night_exposure_vpd",       "Low VPD at night",                 None,                    "low"),
    ("rolling_24_mean_radiation","High radiation load",              "Low radiation",          "both"),
    ("rolling_24_min_ventilation","Low ventilation / poor airflow",  None,                    "low"),
    ("rolling_12_max_rh",        "Peak RH spike (12h)",              None,                    "high"),
    ("rolling_6_mean_temp",      "Temperature anomaly (6h)",         None,                    "high"),
    ("lag_1_vpd",                "Rapid VPD drop",                   None,                    "low"),
    ("temp_x_rh",                "High temp-RH interaction",         None,                    "high"),
    ("vpd_x_rh",                 "High VPD-RH coupling",             None,                    "high"),
    ("dewpoint_spread",          "Narrow dewpoint margin",           None,                    "low"),
]

# Percentile-based thresholds computed from X_train (global reference)
_SIGNAL_HIGH_THRESH: dict = {}
_SIGNAL_LOW_THRESH:  dict = {}

def _build_signal_thresholds(X_train_df: "pd.DataFrame"):
    """Precompute p75 (high) and p25 (low) thresholds from training features."""
    global _SIGNAL_HIGH_THRESH, _SIGNAL_LOW_THRESH
    for substr, *_ in _SIGNAL_PRIORITY:
        cols = [c for c in X_train_df.columns if substr in c]
        if cols:
            vals = X_train_df[cols].values.ravel()
            _SIGNAL_HIGH_THRESH[substr] = float(np.nanpercentile(vals, 75))
            _SIGNAL_LOW_THRESH[substr]  = float(np.nanpercentile(vals, 25))


def extract_dominant_reason(feature_row: "pd.Series", max_signals: int = 3) -> str:
    """
    Given a single-row pandas Series of engineered features, return a
    short reason string (≤3 signals) summarising the top environmental drivers.

    Parameters
    ----------
    feature_row  : pd.Series  — one row from the feature matrix X
    max_signals  : int        — maximum signals to include (default 3)

    Returns
    -------
    str  — e.g. "High RH exposure (24h) + warm nights + low ventilation"
    """
    if feature_row is None:
        return "Environmental stress detected"

    triggered = []
    for substr, label_high, label_low, direction in _SIGNAL_PRIORITY:
        if len(triggered) >= max_signals:
            break
        # Find matching columns in this row
        matching_cols = [c for c in feature_row.index if substr in c]
        if not matching_cols:
            continue

        val = float(np.nanmean([feature_row[c] for c in matching_cols]))

        if direction in ("high", "both"):
            thresh = _SIGNAL_HIGH_THRESH.get(substr)
            if thresh is not None and val >= thresh and label_high:
                triggered.append(label_high)
                continue

        if direction in ("low", "both"):
            thresh = _SIGNAL_LOW_THRESH.get(substr)
            if thresh is not None and val <= thresh:
                label = label_low if label_low else label_high
                if label:
                    triggered.append(label)

    if not triggered:
        return "Adverse environmental conditions detected"

    return " + ".join(triggered)


In [94]:
# ─────────────────────────────────────────────────────────────────────────────
# K2 — generate_high_risk_report()
# Converts raw per-disease risk predictions → actionable alert report.
# ─────────────────────────────────────────────────────────────────────────────

def generate_high_risk_report(
    predicted_risks_dict: dict,
    thresholds_config: dict,
    feature_row=None,
    aggregation: str = "max_next_24h",
    chosen_horizon: int = 24,
) -> dict:
    """
    Generate a structured high-risk alert report from model predictions.

    Parameters
    ----------
    predicted_risks_dict : dict
        {disease: {H: risk_value}} OR {disease: risk_value} (single-horizon).
        If multi-horizon, aggregation mode is applied.
    thresholds_config : dict
        CONFIG['risk_label_bins'] — must contain 'high' and 'medium' keys.
    feature_row : pd.Series or None
        Most recent feature vector for reason extraction.
        If None, a generic reason is generated.
    aggregation : str
        "max_next_24h" (default) — max over H in [6, 12, 24].
        "risk_at_horizon"        — value at chosen_horizon only.
    chosen_horizon : int
        Used only when aggregation == "risk_at_horizon".

    Returns
    -------
    report : dict
        {
          "timestamp"    : str  (UTC ISO-8601 if available),
          "aggregation"  : str,
          "alerts"       : list of dicts  OR  str "No high-risk disease forecast",
          "n_high"       : int,
        }
        Each alert dict:
            {
              "disease"    : str,
              "risk_index" : float  (0–100, rounded 1 dp),
              "risk_label" : "High",
              "reason"     : str,
            }
    """
    HIGH_TH   = thresholds_config["high"][0]    # 67
    MEDIUM_TH = thresholds_config["medium"][0]  # 33

    # ── 1. Compute effective risk value per disease ────────────────────────────
    effective_risk = {}
    for disease, val in predicted_risks_dict.items():
        if isinstance(val, dict):
            # Multi-horizon: {H: risk_value}
            if aggregation == "max_next_24h":
                # Max over horizons ≤ 24 h
                short_H = [h for h in val.keys() if h <= 24]
                effective_risk[disease] = max(val[h] for h in short_H) if short_H \
                                          else max(val.values())
            else:
                # risk_at_horizon: pick closest available
                if chosen_horizon in val:
                    effective_risk[disease] = val[chosen_horizon]
                else:
                    nearest = min(val.keys(), key=lambda h: abs(h - chosen_horizon))
                    effective_risk[disease] = val[nearest]
        else:
            # Already a scalar
            effective_risk[disease] = float(val)

    # Clip to valid range
    effective_risk = {d: float(np.clip(v, 0.0, 100.0))
                      for d, v in effective_risk.items()}

    # ── 2. Filter HIGH diseases, rank by risk index ────────────────────────────
    high_diseases = [(d, v) for d, v in effective_risk.items() if v >= HIGH_TH]
    high_diseases.sort(key=lambda x: x[1], reverse=True)
    top2 = high_diseases[:2]

    # ── 3. Build report ───────────────────────────────────────────────────────
    n_high = len(top2)

    if n_high == 0:
        alerts = "No high-risk disease forecast"
    else:
        alerts = []
        for disease, risk_val in top2:
            reason = extract_dominant_reason(feature_row, max_signals=3)
            alerts.append({
                "disease":    disease,
                "risk_index": round(risk_val, 1),
                "risk_label": "High",
                "reason":     reason,
            })

    report = {
        "aggregation": aggregation,
        "n_high":      n_high,
        "alerts":      alerts,
    }
    return report


def format_report_text(report: dict, timestamp: str = None) -> str:
    """Return a human-readable string version of a report dict."""
    lines = []
    if timestamp:
        lines.append(f"Timestamp : {timestamp}")
    lines.append(f"Mode      : {report['aggregation']}")
    lines.append(f"High risks: {report['n_high']}")
    if isinstance(report["alerts"], str):
        lines.append(f"→ {report['alerts']}")
    else:
        for a in report["alerts"]:
            lines.append(
                f"  ▸ {a['disease']:<22}  Risk={a['risk_index']:5.1f}  "
                f"[{a['risk_label']}]  — {a['reason']}"
            )
    return "\n".join(lines)


In [95]:
# ─────────────────────────────────────────────────────────────────────────────
# K3 — Initialise signal thresholds + demo the report function on 5 samples
# ─────────────────────────────────────────────────────────────────────────────
print("[K3] Building signal thresholds from training features …")
# X is the full engineered feature DataFrame from Section F (build_feature_matrix)
_X_train_df = X.iloc[X.index.isin(
    X_scaled.index[:len(rf_scaled[6]["X_train"])]   # train indices
)]
# Fallback: use full X if index slicing is uncertain
_X_train_df = X.iloc[:len(rf_scaled[6]["X_train"])]
_build_signal_thresholds(_X_train_df)
print(f"     Thresholds built for {len(_SIGNAL_HIGH_THRESH)} signal groups.")

# ── Demo: 5 test samples from LSTM predictions ────────────────────────────────
print("\n[K3] Demo — 5 test-set reports (LSTM, aggregation=max_next_24h):")
print("─" * 70)

X_test_df  = X.iloc[-len(lstm_tensors["test"]["X"]):]  # test-set feature rows
preds_test = lstm_model.predict(lstm_tensors["test"]["X"], verbose=0)

DEMO_INDICES = [0, 24, 72, 120, 167]
for idx in DEMO_INDICES:
    # Build per-disease multi-horizon risk dict
    risks_dict = {}
    for d_idx, disease in enumerate(CONFIG["diseases"]):
        risks_dict[disease] = {
            H: float(preds_test[f"output_H{H}"][idx, d_idx])
            for H in CONFIG["horizon_H"]
        }

    feat_row = X_test_df.iloc[idx] if idx < len(X_test_df) else None
    ts_str   = str(X_test_df.index[idx]) if idx < len(X_test_df) else f"idx={idx}"

    report = generate_high_risk_report(
        predicted_risks_dict=risks_dict,
        thresholds_config=CONFIG["risk_label_bins"],
        feature_row=feat_row,
        aggregation="max_next_24h",
    )
    print(format_report_text(report, timestamp=ts_str))
    print("─" * 70)

print("[K3] Demo complete.")


[K3] Building signal thresholds from training features …
     Thresholds built for 3 signal groups.

[K3] Demo — 5 test-set reports (LSTM, aggregation=max_next_24h):
──────────────────────────────────────────────────────────────────────
Timestamp : 2025-01-03 00:00:00
Mode      : max_next_24h
High risks: 2
  ▸ powdery_mildew          Risk=100.0  [High]  — Narrow dewpoint margin
  ▸ early_blight            Risk= 76.3  [High]  — Narrow dewpoint margin
──────────────────────────────────────────────────────────────────────
Timestamp : 2025-01-04 00:00:00
Mode      : max_next_24h
High risks: 1
  ▸ powdery_mildew          Risk=100.0  [High]  — Narrow dewpoint margin
──────────────────────────────────────────────────────────────────────
Timestamp : 2025-01-06 00:00:00
Mode      : max_next_24h
High risks: 1
  ▸ powdery_mildew          Risk=100.0  [High]  — Narrow dewpoint margin
──────────────────────────────────────────────────────────────────────
Timestamp : 2025-01-08 00:00:00
Mode      : m

## Section L — Complete Artifact Export

Consolidates all outputs from Sections A–K into a single, reproducible artifact set.

### Complete artifact manifest
| Artifact | Location | Format |
|---|---|---|
| LSTM model (best weights) | `src/agritwin_gh/models/disease_progression/lstm_<run_id>.keras` | Keras SavedModel |
| RF model (per horizon) | `src/agritwin_gh/models/dp_rf_H{H}_<run_id>.joblib` | joblib |
| Feature scaler | `artifacts/dp_<run_id>/scaler.pkl` | pickle |
| Feature schema | `artifacts/dp_<run_id>/feature_schema.json` | JSON |
| Thresholds config | `artifacts/dp_<run_id>/thresholds_config.json` | JSON |
| Full evaluation report | `artifacts/dp_<run_id>/evaluation_report.json` | JSON |
| Evaluation summary (CSV) | `artifacts/dp_<run_id>/evaluation_report_summary.csv` | CSV |
| 20+ sample reports | `artifacts/dp_<run_id>/final_sample_reports.json` | JSON |
| RF training history (CSV) | `artifacts/dp_<run_id>/rnn_training_history.csv` | CSV |
| All plots (*.png) | `artifacts/dp_<run_id>/` | PNG 150 dpi |
| Artifact manifest | `artifacts/dp_<run_id>/artifact_manifest.json` | JSON |


In [100]:
# ─────────────────────────────────────────────────────────────────────────────
# L1 — compile_evaluation_report()
# Merges RF + LSTM metrics into a single structured evaluation report dict.
# ─────────────────────────────────────────────────────────────────────────────
import json, csv as _csv
from datetime import datetime, timezone

def compile_evaluation_report(
    rf_metrics_dict,
    lstm_metrics_dict,
    config,
) -> dict:
    """
    Build a unified evaluation report combining RF and LSTM metrics.

    Returns
    -------
    report : dict  — JSON-serialisable
    """
    diseases  = config["diseases"]
    horizon_H = config["horizon_H"]
    run_id    = config["run_id"]

    def _round_floats(obj):
        if isinstance(obj, dict):
            return {str(k): _round_floats(v) for k, v in obj.items()}
        if isinstance(obj, (float, np.floating)):
            return round(float(obj), 6)
        return obj

    report = {
        "meta": {
            "run_id":        run_id,
            "generated_utc": datetime.now(timezone.utc).isoformat(),
            "diseases":      diseases,
            "horizon_H":     horizon_H,
            "high_threshold":  config["risk_label_bins"]["high"][0],
            "medium_threshold":config["risk_label_bins"]["medium"][0],
        },
        "models": {
            "RF": {
                "type": "MultiOutputRegressor(RandomForestRegressor)",
                "n_models": len(horizon_H),
                "params":   config["rf_params"],
                "metrics":  _round_floats(rf_metrics_dict),
            },
            "LSTM": {
                "type":    "Multi-head LSTM (shared encoder)",
                "n_heads": len(horizon_H),
                "params":  config["lstm"],
                "metrics": _round_floats(lstm_metrics_dict),
            },
        },
    }

    # ── Summary table: mean test MAE per model × horizon ──────────────────────
    summary = {}
    for H in horizon_H:
        rf_mae_mean   = np.mean([rf_metrics_dict[H]["test"][d]["MAE"]   for d in diseases])
        lstm_mae_mean = np.mean([lstm_metrics_dict[H]["test"][d]["MAE"] for d in diseases])
        rf_f1_mean    = np.mean([rf_metrics_dict[H]["test"][d]["F1_High"]   for d in diseases])
        lstm_f1_mean  = np.mean([lstm_metrics_dict[H]["test"][d]["F1_High"] for d in diseases])
        summary[f"H{H}"] = {
            "RF_mean_MAE":   round(float(rf_mae_mean),   4),
            "LSTM_mean_MAE": round(float(lstm_mae_mean), 4),
            "RF_F1_High":    round(float(rf_f1_mean),    4),
            "LSTM_F1_High":  round(float(lstm_f1_mean),  4),
        }

    report["summary_table"] = summary

    # ── Print summary ─────────────────────────────────────────────────────────
    print("[L1] Evaluation report compiled.")
    print(f"\n     {'Horizon':<8}  {'RF MAE':>8}  {'LSTM MAE':>10}  "
          f"{'RF F1-H':>8}  {'LSTM F1-H':>10}")
    print(f"     {'─' * 54}")
    for H in horizon_H:
        s = summary[f"H{H}"]
        print(f"     H={H:<5}    {s['RF_mean_MAE']:>8.4f}  {s['LSTM_mean_MAE']:>10.4f}  "
              f"{s['RF_F1_High']:>8.4f}  {s['LSTM_F1_High']:>10.4f}")

    return report


def save_evaluation_report(report: dict, out_dir) -> dict:
    """Save evaluation_report.json + evaluation_report_summary.csv."""
    saved = {}

    # JSON
    json_path = out_dir / "evaluation_report.json"
    with open(json_path, "w") as f:
        json.dump(report, f, indent=2)
    saved["evaluation_report_json"] = str(json_path)

    # CSV summary table
    rows = []
    for h_key, vals in report["summary_table"].items():
        rows.append({"horizon": h_key, **vals})
    csv_path = out_dir / "evaluation_report_summary.csv"
    with open(csv_path, "w", newline="") as f:
        writer = _csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    saved["evaluation_report_csv"] = str(csv_path)

    print(f"  [L1] evaluation_report.json  →  {json_path.name}")
    print(f"  [L1] evaluation_report_summary.csv  →  {csv_path.name}")
    return saved


In [101]:
# ─────────────────────────────────────────────────────────────────────────────
# L2 — run_sample_reports()
# Runs generate_high_risk_report() on ≥20 evenly-spaced test timestamps
# using LSTM predictions, and saves results to final_sample_reports.json.
# ─────────────────────────────────────────────────────────────────────────────

def run_sample_reports(
    lstm_model,
    lstm_tensors,
    X_feature_df,
    config,
    n_samples: int = 20,
    aggregation: str = "max_next_24h",
    out_dir=None,
) -> list:
    """
    Generate and save sample high-risk reports for a selection of test samples.

    Parameters
    ----------
    lstm_model      : trained Keras model
    lstm_tensors    : dict from J1
    X_feature_df    : pd.DataFrame — full engineered feature matrix (Section F)
    config          : CONFIG dict
    n_samples       : int — number of evenly-spaced samples (default 20)
    aggregation     : str — "max_next_24h" | "risk_at_horizon"
    out_dir         : pathlib.Path — where to save final_sample_reports.json

    Returns
    -------
    sample_reports : list of dicts  (one per sample)
    """
    import json as _json

    out_dir = out_dir or P_ARTIFACTS_DP
    diseases  = config["diseases"]
    horizon_H = config["horizon_H"]

    # Run predictions on test set once
    X_test = lstm_tensors["test"]["X"]
    n_test = X_test.shape[0]
    preds  = lstm_model.predict(X_test, verbose=0)  # {output_HH: (n, D)}

    # Test-set feature rows (for reason extraction)
    n_feat_test = len(X_feature_df) - n_test
    X_test_df   = X_feature_df.iloc[n_feat_test:].reset_index(drop=False)

    # Evenly spaced indices over test set
    step    = max(1, n_test // n_samples)
    indices = list(range(0, min(n_test, step * n_samples), step))[:n_samples]

    sample_reports = []
    high_count = 0

    for i, idx in enumerate(indices):
        # Build multi-horizon risk dict
        risks_dict = {}
        for d_idx, disease in enumerate(diseases):
            risks_dict[disease] = {
                H: float(np.clip(preds[f"output_H{H}"][idx, d_idx], 0.0, 100.0))
                for H in horizon_H
            }

        ts_str   = str(X_test_df["index"].iloc[idx]) \
                   if "index" in X_test_df.columns and idx < len(X_test_df) \
                   else f"test_idx_{idx}"
        feat_row = X_test_df.drop(columns=["index"], errors="ignore").iloc[idx] \
                   if idx < len(X_test_df) else None

        report = generate_high_risk_report(
            predicted_risks_dict=risks_dict,
            thresholds_config=config["risk_label_bins"],
            feature_row=feat_row,
            aggregation=aggregation,
        )
        report["timestamp"]      = ts_str
        report["sample_index"]   = int(idx)
        report["raw_risks"]      = {
            d: {str(H): round(v, 2) for H, v in risks_dict[d].items()}
            for d in diseases
        }
        sample_reports.append(report)
        if report["n_high"] > 0:
            high_count += 1

    # Save to JSON
    out_path = out_dir / "final_sample_reports.json"
    with open(out_path, "w") as f:
        _json.dump(sample_reports, f, indent=2, default=str)

    print(f"[L2] Sample reports generated: {len(sample_reports)}  "
          f"({high_count} with HIGH alerts)")
    print(f"     Saved → {out_path.name}")

    # Print a few for inspection
    print("\n[L2] First 3 reports:")
    for rep in sample_reports[:3]:
        print(format_report_text(rep, timestamp=rep.get("timestamp")))
        print("·" * 50)

    return sample_reports, out_path


In [102]:
# ─────────────────────────────────────────────────────────────────────────────
# L3 — export_all_artifacts()
# Saves thresholds_config.json + artifact_manifest.json; registers all known
# artifact paths from previous sections into one consolidated manifest.
# ─────────────────────────────────────────────────────────────────────────────

def export_all_artifacts(
    config,
    rf_saved_paths_dict,
    lstm_saved_paths_dict,
    eval_report_paths_dict,
    sample_report_path,
    feat_artifact_paths_dict,
    out_dir=None,
) -> dict:
    """
    Write thresholds_config.json and build the complete artifact_manifest.json.

    Returns
    -------
    manifest : dict  — complete artifact manifest
    """
    import json as _json
    from datetime import datetime, timezone

    out_dir = out_dir or P_ARTIFACTS_DP

    # ── 1. thresholds_config.json ─────────────────────────────────────────────
    thr_path = out_dir / "thresholds_config.json"
    thr_data = {
        "risk_label_bins":    config["risk_label_bins"],
        "disease_thresholds": config.get("disease_thresholds", {}),
        "horizon_H":          config["horizon_H"],
        "diseases":           config["diseases"],
        "run_id":             config["run_id"],
    }
    with open(thr_path, "w") as f:
        _json.dump(thr_data, f, indent=2)
    print(f"  [L3] thresholds_config.json saved.")

    # ── 2. Collect all known artifact paths ───────────────────────────────────
    manifest = {
        "run_id":        config["run_id"],
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        # Model files
        "models": {
            "lstm": lstm_saved_paths_dict.get("lstm_model", "NOT_FOUND"),
            "rf":   {f"H{H}": rf_saved_paths_dict.get(f"rf_model_H{H}", "NOT_FOUND")
                     for H in config["horizon_H"]},
        },
        # Scalers + schema
        "preprocessing": {
            k: str(v) for k, v in feat_artifact_paths_dict.items()
        },
        # Reports + metrics
        "evaluation": {
            **{k: str(v) for k, v in eval_report_paths_dict.items()},
            "rf_metrics_json":   rf_saved_paths_dict.get("rf_metrics_json",  "NOT_FOUND"),
            "rf_metrics_csv":    rf_saved_paths_dict.get("rf_metrics_csv",   "NOT_FOUND"),
            "rnn_metrics_json":  lstm_saved_paths_dict.get("rnn_metrics_json","NOT_FOUND"),
            "rnn_metrics_csv":   lstm_saved_paths_dict.get("rnn_metrics_csv", "NOT_FOUND"),
        },
        # Sample reports
        "sample_reports": str(sample_report_path),
        # Thresholds
        "thresholds_config": str(thr_path),
        # Plots
        "plots": {
            k: str(v) for k, v in {
                **{k: v for k, v in rf_saved_paths_dict.items()  if "plot" in k},
                **{k: v for k, v in lstm_saved_paths_dict.items() if "plot" in k},
            }.items()
        },
    }

    # ── 3. Save manifest ──────────────────────────────────────────────────────
    manifest_path = out_dir / "artifact_manifest.json"
    with open(manifest_path, "w") as f:
        _json.dump(manifest, f, indent=2)
    print(f"  [L3] artifact_manifest.json saved.")

    # ── 4. Pretty-print manifest ──────────────────────────────────────────────
    print(f"\n[L3] Complete artifact manifest for run_id={config['run_id']}:")
    def _print_nested(d, indent=0):
        for k, v in d.items():
            if isinstance(v, dict):
                print(f"  {'  ' * indent}{k}:")
                _print_nested(v, indent + 1)
            else:
                short = str(v).replace(str(_ROOT), ".") if str(_ROOT) in str(v) else str(v)
                print(f"  {'  ' * indent}{k:<30} → {short}")
    _print_nested(manifest)

    return manifest


In [103]:
# ─────────────────────────────────────────────────────────────────────────────
# L4 — Run complete export pipeline + final summary
#
# HOW TO REPRODUCE (IEEE-friendly)
# ─────────────────────────────────────────────────────────────────────────────
# Dataset
#   Place processed Greenhouse Indoor Conditions CSV files in:
#     data/processed/Disease/
#   Expected columns (minimum): timestamp, temperature_celsius,
#   relative_humidity_pct, vpd_kPa, radiation_Wm2, ventilation_rate,
#   co2_ppm, day_night_flag, <disease>_binary for each disease.
#
# Environment
#   python -m venv .venv && .venv/Scripts/activate   # Windows
#   pip install -r requirements.txt                  # TF 2.x, scikit-learn, …
#
# Run order (Kernel → Run All or execute sections in order):
#   A  — CONFIG, seeds, paths
#   B  — Data loading + validation
#   C  — Chronological split (70/15/15)
#   D  — Disease thresholds + risk index computation
#   E  — Baseline artifact save
#   F  — Feature engineering (rolling / lag / interaction / cyclical)
#   G  — Supervised dataset building (RF tabular + RNN windows)
#   H  — StandardScaler fit + feature schema artifacts
#   I  — RF training + evaluation + plots
#   J  — LSTM training + evaluation + plots        ← this section
#   K  — High-risk report logic
#   L  — Complete artifact export                  ← this cell
#
# Outputs
#   Models  : src/agritwin_gh/models/
#   Artifacts: src/agritwin_gh/models/artifacts/dp_<run_id>/
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("SECTION L: COMPLETE ARTIFACT EXPORT")
print("=" * 70)

_t_l = _time_mod.time()

# ── L1: Compile evaluation report ────────────────────────────────────────────
print("\n[L1] Compiling unified RF + LSTM evaluation report …")
eval_report = compile_evaluation_report(rf_metrics, lstm_metrics, CONFIG)
eval_report_paths = save_evaluation_report(eval_report, P_ARTIFACTS_DP)

# ── L2: Run 20 sample reports ─────────────────────────────────────────────────
print("\n[L2] Running sample reports on 20+ test timestamps …")
sample_reports, sample_report_path = run_sample_reports(
    lstm_model=lstm_model,
    lstm_tensors=lstm_tensors,
    X_feature_df=X,
    config=CONFIG,
    n_samples=20,
    aggregation="max_next_24h",
    out_dir=P_ARTIFACTS_DP,
)

# ── L3: Export all artifacts + write manifest ─────────────────────────────────
print("\n[L3] Exporting all artifacts and writing manifest …")
final_manifest = export_all_artifacts(
    config=CONFIG,
    rf_saved_paths_dict=rf_saved_paths,
    lstm_saved_paths_dict=lstm_saved_paths,
    eval_report_paths_dict=eval_report_paths,
    sample_report_path=sample_report_path,
    feat_artifact_paths_dict=feat_artifact_paths,
    out_dir=P_ARTIFACTS_DP,
)

# ── Final timing summary ──────────────────────────────────────────────────────
elapsed_l   = _time_mod.time() - _t_l
total_final = _time_mod.time() - _t0_total

print(f"\n{'=' * 70}")
print("DISEASE PROGRESSION RISK FORECASTING — PIPELINE COMPLETE")
print(f"{'=' * 70}")
print(f"Run ID          : {CONFIG['run_id']}")
print(f"Section L time  : {elapsed_l:.1f}s")
print(f"Total wall time : {total_final:.1f}s  ({total_final/60:.1f} min)")
print(f"Artifacts dir   : {P_ARTIFACTS_DP.relative_to(_ROOT)}")
print(f"Models dir      : {P_LSTM_MODEL_DIR.relative_to(_ROOT)}")
print(f"{'─' * 70}")

# ── List artifacts directory ───────────────────────────────────────────────────
_arts = sorted(P_ARTIFACTS_DP.iterdir())
print(f"\nArtifacts ({len(_arts)} files):")
for p in _arts:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<50}  {size_kb:>8.1f} KB")

_models = sorted(P_MODELS_DIR.rglob("dp_*"))
print(f"\nModel files ({len(_models)} files):")
for p in _models:
    size_mb = p.stat().st_size / 1_048_576
    print(f"  {str(p.relative_to(P_MODELS_DIR)):<50}  {size_mb:>8.2f} MB")

print(f"\n{'=' * 70}")
print("AgriTwin-GH Disease Progression module — all sections complete.")
print(f"{'=' * 70}")


SECTION L: COMPLETE ARTIFACT EXPORT

[L1] Compiling unified RF + LSTM evaluation report …
[L1] Evaluation report compiled.

     Horizon     RF MAE    LSTM MAE   RF F1-H   LSTM F1-H
     ──────────────────────────────────────────────────────
     H=6          2.2762      4.2695    0.9522      0.7481
     H=12         3.6676      4.9889    0.9342      0.7352
     H=24        10.1040     12.1479    0.7542      0.4928
     H=48        17.7138     15.5250    0.4166      0.4163
  [L1] evaluation_report.json  →  evaluation_report.json
  [L1] evaluation_report_summary.csv  →  evaluation_report_summary.csv

[L2] Running sample reports on 20+ test timestamps …
[L2] Sample reports generated: 20  (19 with HIGH alerts)
     Saved → final_sample_reports.json

[L2] First 3 reports:
Timestamp : test_idx_0
Mode      : max_next_24h
High risks: 2
  ▸ powdery_mildew          Risk=100.0  [High]  — Narrow dewpoint margin
  ▸ early_blight            Risk= 76.3  [High]  — Narrow dewpoint margin
·············